# Data Loaders
> Unified data loading pipelines integrating fastai DataBlocks with MONAI Datasets.

The `bioMONAI.data.load` module provides a highly flexible, registry-based architecture for building complex biomedical imaging pipelines. It seamlessly bridges the gap between MONAI's powerful, memory-efficient datasets (like `CacheDataset` or `PersistentDataset`) and fastai's intuitive training ecosystem (`DataLoaders`, `DataBlock`), offering automated source detection, prefix-based configuration routing, and smart dataset splitting.

In [ ]:
#| default_exp data.load

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export

# =================================
# Standard library
# =================================
import os
import types
from inspect import signature, _empty
from pathlib import Path
import pandas as pd

# for pipeline context
from dataclasses import dataclass, field
from typing import Any, Optional

# =================================
# PyTorch
# =================================
from torch.utils.data import DataLoader as torchDataLoader
from torch.utils.data import Dataset as torchDataset

# =================================
# fastai
# =================================
from fastai.data.all import (
    DataLoaders, delegates, RegexLabeller, is_listy,
    ColReader
)

from fastai.vision.all import (
    DataBlock, CategoryBlock, MultiCategoryBlock, RegressionBlock,
    TfmdDL, TransformBlock, Pipeline,
    get_image_files, 
    parent_label, 
    partial, show_results, store_attr
)

# =================================
# MONAI
# =================================
from monai.data import Dataset as MonaiDataset, CacheDataset, PersistentDataset, SmartCacheDataset
# from monai.data.utils import pickle_hashing
from monai.transforms import Compose
from monai.transforms.transform import Randomizable

# =================================
# bioMONAI
# =================================
from bioMONAI.utils import *
from bioMONAI.data.core import *
# from bioMONAI.transforms import RandMonaiTransform, RandTransform

# =================================
# fasttransform patch
# =================================
import fasttransform

_original_repr = fasttransform.transform.Transform.__repr__

def _safe_repr(self):
    try:
        return _original_repr(self)
    except AttributeError as e:
        if "'_BoundFunction' object has no attribute 'methods'" in str(e):
            return f"{self.__class__.__name__}(...)"
        raise

fasttransform.transform.Transform.__repr__ = _safe_repr

## BioDataBlocks

The **`BioDataBlock`** class is built on top of the `DataBlock` class provided by the fastai library. It serves as a tailored, high-level container used to build datasets and dataloaders from blocks specifically designed for biomedical data. By doing so, it seamlessly integrates item and batch transformations, specialized input getters, and splitting logic into a single cohesive pipeline.

In [ ]:
#| export
class BioDataBlock(DataBlock):
    """ 
    The `BioDataBlock` class serves as a generic container to build `Datasets` and `DataLoaders` efficiently. It integrates item and batch transformations, getters, and splitters, simplifying the setup of data pipelines for training and validation.
    """
    def __init__(self, 
            blocks:list=(BioImage.get_datablock(), BioImage.get_datablock()),       # One or more `TransformBlock`s
            dl_type:TfmdDL=None,                                                    # Task specific `TfmdDL`, defaults to `block`'s dl_type or`TfmdDL`
            get_items=get_image_files,
            get_y=None,
            get_x=None,
            getters:list=None,                                                      # Getter functions applied to results of `get_items`
            n_inp:int=None,                                                         # Number of inputs
            item_tfms:list=None,                                                    # `ItemTransform`s, applied on an item 
            batch_tfms:list=None,                                                   # `Transform`s or `RandTransform`s, applied by batch
            splitter=None, 
        ):
        super().__init__(
            blocks=blocks, 
            dl_type=dl_type, 
            get_items=get_items,
            get_y=get_y,
            get_x=get_x,
            getters=getters, 
            n_inp=n_inp, 
            item_tfms=item_tfms, 
            batch_tfms=batch_tfms,
            splitter=splitter,
            )
        

In [ ]:
# Example: Initializing a BioDataBlock
# Creates a preconfigured dataset builder tailored for biomedical imaging
bio_block = BioDataBlock(n_inp=1)

print(f"Number of inputs: {bio_block.n_inp}")
print(f"Default blocks assigned: {len(bio_block.blocks)}")

Number of inputs: 1
Default blocks assigned: 2


In [ ]:
#| hide
from fastcore.test import test_eq
from fastai.vision.all import DataBlock

def test_biodatablock():
    # Init
    builder = BioDataBlock(n_inp=1)
    test_eq(isinstance(builder, DataBlock), True)
    test_eq(builder.n_inp, 1)

    # Verify default blocks
    default_builder = BioDataBlock()
    test_eq(len(default_builder.blocks), 2)
    
    return "BioDataBlock tests passed"

test_eq(test_biodatablock(), "BioDataBlock tests passed")

## BioDataloaders: unified method

The module offers classes to construct data loaders for `fastTrainer`, seamlessly supporting different dataset backends.

### Component Registries

These dynamic registries (`SOURCE_REGISTRY`, `DATASET_REGISTRY`, `LOADER_REGISTRY`, `TASK_REGISTRY`) allow you to easily add new components—such as custom data sources, specialized datasets, loaders, or task handlers—without modifying the core library code.

In [ ]:
#| export
SOURCE_REGISTRY = {}
DATASET_REGISTRY = {}
LOADER_REGISTRY = {}
TASK_REGISTRY = {}

def register_source(name):
    def wrapper(cls):
        SOURCE_REGISTRY[name] = cls
        return cls
    return wrapper


def register_dataset(name, backend):
    def wrapper(cls):
        DATASET_REGISTRY[name] = (cls, backend)
        return cls
    return wrapper


def register_loader(name):
    def wrapper(cls):
        LOADER_REGISTRY[name] = cls
        return cls
    return wrapper

def register_task(name):
    def wrapper(cls):
        TASK_REGISTRY[name] = cls
        return cls
    return wrapper

In [ ]:
# Example: Dynamically registering custom components

@register_source("my_custom_source")
class MyCustomSource:
    pass

@register_dataset("my_custom_dataset", backend="pytorch")
class MyCustomDataset:
    pass

# Verify components are in the global registries
print(f"Available sources: {list(SOURCE_REGISTRY.keys())}")
print(f"Available datasets: {list(DATASET_REGISTRY.keys())}")

Available sources: ['my_custom_source']
Available datasets: ['my_custom_dataset']


In [ ]:
#| hide
from fastcore.test import test_eq

def test_registries():
    # Register dummy components
    @register_source("dummy_src")
    class DummySrc: pass

    @register_dataset("dummy_ds", backend="dummy")
    class DummyDs: pass

    @register_loader("dummy_ld")
    class DummyLd: pass

    @register_task("dummy_tk")
    class DummyTk: pass

    # Verify registration
    test_eq(SOURCE_REGISTRY["dummy_src"], DummySrc)
    test_eq(DATASET_REGISTRY["dummy_ds"], (DummyDs, "dummy"))
    test_eq(LOADER_REGISTRY["dummy_ld"], DummyLd)
    test_eq(TASK_REGISTRY["dummy_tk"], DummyTk)

    # Cleanup
    del SOURCE_REGISTRY["dummy_src"]
    del DATASET_REGISTRY["dummy_ds"]
    del LOADER_REGISTRY["dummy_ld"]
    del TASK_REGISTRY["dummy_tk"]

    if "my_custom_source" in SOURCE_REGISTRY:
        del SOURCE_REGISTRY["my_custom_source"]
    if "my_custom_dataset" in DATASET_REGISTRY:
        del DATASET_REGISTRY["my_custom_dataset"]
        
    return "Registry tests passed"

test_eq(test_registries(), "Registry tests passed")

### Utility Functions

This section provides specialized helpers to bridge configurations and data formats seamlessly across the pipeline.

**`split_prefixed_kwargs`** is a utility designed to cleanly separate a single dictionary of keyword arguments into multiple configuration groups based on string prefixes (such as `train_` and `val_`). Arguments with a matching prefix are routed to their respective group, while non-prefixed arguments are automatically applied to all groups as fallback defaults. This is highly useful for managing distinct parameters for Training and Validation DataLoaders from a single unified config.

**`ReadDictDataset`** acts as a crucial bridge between MONAI's dictionary-based datasets and standard PyTorch/fastai pipelines, which expect tuple-based inputs. It wraps a dictionary dataset and flattens the specified input (`x_keys`) and target (`y_keys`) dictionaries into a single, ordered tuple upon item retrieval.

In [ ]:
#| export
def split_prefixed_kwargs(kwargs, prefixes=("train_", "val_")):
    """
    Split a dictionary of kwargs into multiple groups based on prefixes.

    Example:
        kwargs = {
            "batch_size": 32,
            "train_cache_rate": 1.0,
            "val_cache_rate": 0.5
        }

        split_prefixed_kwargs(kwargs)
        -> {
             "train": {"cache_rate": 1.0, "batch_size": 32},
             "val": {"cache_rate": 0.5, "batch_size": 32}
           }

    Rules:
    - Keys starting with prefix go to that group (prefix removed)
    - All keys also appear in each group as defaults if no prefix exists
    """
    groups = {p.rstrip("_"): {} for p in prefixes}

    for k, v in kwargs.items():
        matched = False
        for p in prefixes:
            if k.startswith(p):
                groups[p.rstrip("_")][k[len(p):]] = v
                matched = True
                break
        if not matched:
            # No prefix → default to all groups
            for g in groups:
                groups[g][k] = v

    return groups

In [ ]:
# Example: Splitting mixed keyword arguments into specific groups

sample_kwargs = {
    "batch_size": 32,
    "train_cache_rate": 1.0,
    "val_cache_rate": 0.5,
    "shuffle": True,
    "val_shuffle": False
}

# Split based on prefixes
split_results = split_prefixed_kwargs(sample_kwargs, prefixes=("train_", "val_"))

print("Train parameters:", split_results["train"])
print("Val parameters:  ", split_results["val"])

Train parameters: {'batch_size': 32, 'cache_rate': 1.0, 'shuffle': True}
Val parameters:   {'batch_size': 32, 'cache_rate': 0.5, 'shuffle': False}


In [ ]:
#| hide
from fastcore.test import test_eq

def test_split_prefixed_kwargs():
    test_kwargs = {
        "batch_size": 32,
        "train_cache_rate": 1.0,
        "val_cache_rate": 0.5,
        "shuffle": True,
        "val_shuffle": False
    }

    # Split
    res = split_prefixed_kwargs(test_kwargs, prefixes=("train_", "val_"))

    # Verify train group
    test_eq(res["train"], {"batch_size": 32, "shuffle": True, "cache_rate": 1.0})
    
    # Verify val group
    test_eq(res["val"], {"batch_size": 32, "shuffle": False, "cache_rate": 0.5})

    return "split_prefixed_kwargs tests passed"

test_eq(test_split_prefixed_kwargs(), "split_prefixed_kwargs tests passed")

In [ ]:
#| export
class ReadDictDataset(torchDataset):
    def __init__(self, ds, x_keys="image", y_keys="label"):
        """
        ds: MONAI dataset (or any dict-like dataset)
        x_keys: single key (str) or list/tuple of keys for inputs
        y_keys: single key (str) or list/tuple of keys for outputs
        """
        self.ds = ds
        # Normalize to lists
        self.x_keys = [x_keys] if isinstance(x_keys, str) else list(x_keys)
        self.y_keys = [y_keys] if isinstance(y_keys, str) else list(y_keys)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        # Flatten all keys into a single tuple
        output = tuple(item[k] for k in self.x_keys + self.y_keys)
        return output

    def __getattr__(self, name):
        # Forward any unknown attribute to the underlying dataset
        return getattr(self.ds, name)

In [ ]:
# Example: Converting a dictionary dataset into a tuple-based dataset

# 1. Mock dictionary dataset
mock_dict_dataset = [
    {"image": "scan_01.nii.gz", "mask": "seg_01.nii.gz", "meta": "patient_A"},
    {"image": "scan_02.nii.gz", "mask": "seg_02.nii.gz", "meta": "patient_B"}
]

# 2. Wrap to flatten specific keys into a tuple
tuple_dataset = ReadDictDataset(
    ds=mock_dict_dataset, 
    x_keys=["image", "meta"], 
    y_keys="mask"
)

# 3. Retrieve item
print(f"Original: {mock_dict_dataset[0]}")
print(f"Wrapped:  {tuple_dataset[0]}")

Original: {'image': 'scan_01.nii.gz', 'mask': 'seg_01.nii.gz', 'meta': 'patient_A'}
Wrapped:  ('scan_01.nii.gz', 'patient_A', 'seg_01.nii.gz')


In [ ]:
#| hide
from fastcore.test import test_eq

def test_readdictdataset():
    # Dummy dict dataset
    class DummyDS:
        def __init__(self):
            self.data = [{"img": "i1.png", "lbl": 0, "ext": "a"}, {"img": "i2.png", "lbl": 1, "ext": "b"}]
            self.attr = "ok"
        def __len__(self): return len(self.data)
        def __getitem__(self, i): return self.data[i]

    dummy_ds = DummyDS()

    # Single keys
    ds_single = ReadDictDataset(dummy_ds, x_keys="img", y_keys="lbl")
    test_eq(len(ds_single), 2)
    test_eq(ds_single[0], ("i1.png", 0))

    # Multiple keys
    ds_multi = ReadDictDataset(dummy_ds, x_keys=["img", "ext"], y_keys=["lbl"])
    test_eq(ds_multi[1], ("i2.png", "b", 1))

    # Delegate attributes
    test_eq(ds_single.attr, "ok")

    return "ReadDictDataset tests passed"

test_eq(test_readdictdataset(), "ReadDictDataset tests passed")

In [ ]:
#| export
def _patch_dataset(ds, **attrs):
    """
    Patch a dataset instance with arbitrary attributes.

    This allows adding fastai-style attributes (like `vocab`) or other
    metadata to a dataset instance without modifying the class.

    Parameters
    ----------
    ds : Dataset
        Dataset instance to patch.
    **attrs : dict
        Arbitrary attributes to attach to the dataset.

    Returns
    -------
    Dataset
        The patched dataset.
    """
    for k, v in attrs.items():
        setattr(ds, k, v)
    return ds

In [ ]:
#| hide
from fastcore.test import test_eq

def test_patch_dataset():
    class DummyDS: pass
    test_ds = DummyDS()

    # Patch attributes
    patched = _patch_dataset(test_ds, vocab=["A", "B"], num_classes=2, is_patched=True)

    # Verify
    test_eq(patched.vocab, ["A", "B"])
    test_eq(patched.num_classes, 2)
    test_eq(patched.is_patched, True)

    return "_patch_dataset tests passed"

test_eq(test_patch_dataset(), "_patch_dataset tests passed")

In [ ]:
#| export
def _patch_dataloader(dl):
    """
    Patch a PyTorch DataLoader for minimal fastai compatibility.

    Adds:
    - .one_batch(): returns a single batch
    - .new(**kwargs): clone dataloader with overrides
    - .show_results(): show batch results using BioImage/MetaTensor
    - .show_batch(): show a batch using BioImage/MetaTensor
    - .vocab: inferred from underlying dataset if present
    """

    # ---- helpers ----
    def _attach_method(obj, fn):
        setattr(obj, fn.__name__, types.MethodType(fn, obj))
    
    def _get_batch_items(b, max_n):
        "Return first `max_n` items from batch `b`, handling single tensors."
        # unpack batch
        if isinstance(b, (tuple, list)) and len(b) == 2:
            x, y = b
        else:
            x = b
            y = None

        # ensure x_items is a list
        if isinstance(x, MetaTensor):
            if x.dim() == 0:          # scalar
                x_items = [x]
            elif x.dim() == 3:        # single image (C,H,W)
                x_items = [x]
            else:                     # batched (B,C,H,W)
                x_items = [x[i] for i in range(min(max_n, x.shape[0]))]
        elif isinstance(x, list):
            x_items = x[:max_n]
        else:
            x_items = list(x)[:max_n]

        # same for y
        if y is None:
            y_items = [None]*len(x_items)
        elif isinstance(y, MetaTensor):
            if y.dim() == 0:
                y_items = [y.item()]
            elif y.dim() == 1:
                y_items = [y[i].item() for i in range(min(max_n, len(y)))]
            else:
                y_items = [y[i] for i in range(min(max_n, y.shape[0]))]
        else:
            y_items = list(y)[:len(x_items)]

        return x_items, y_items

    # ---- methods ----

    def do_item(self, i):
        """
        Return a single item from the dataset after minimal processing.
        Mimics fastai DataLoader.do_item behavior.
        """
        item = self.dataset[i]

        # if collate_fn exists, apply it to make batch-like
        if getattr(self, "collate_fn", None) is not None:
            try:
                item = self.collate_fn([item])
            except:
                pass

        return item

    def one_batch(self): return next(iter(self))

    def new(self, **kwargs):
        params = dict(
            dataset=self.dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            collate_fn=self.collate_fn,
            pin_memory=self.pin_memory,
            drop_last=self.drop_last,
        )
        params.update(kwargs)
        new_dl = type(self)(**params)
        _patch_dataloader(new_dl)
        return new_dl

    # alias the typedispatch function
    from bioMONAI.data import show_batch as _show_batch

    def show_batch(
        self, b=None, max_n=9, ctxs=None, show=True, unique=False, **kwargs
    ):
        "Show `max_n` input(s) and target(s) from the batch."
        if unique:
            old_get_idxs = getattr(self, "get_idxs", lambda: None)
            self.get_idxs = lambda: [0]

        if b is None: b = self.one_batch()
        x_items, y_items = _get_batch_items(b, max_n)

        if show:
            _show_batch(
                x_items,
                y_items,
                samples=None,
                ctxs=ctxs,
                max_n=max_n,
                vocab=getattr(self, "vocab", None),
                **kwargs
            )
        else:
            return x_items, y_items

        if unique: self.get_idxs = old_get_idxs

    # ---- attach methods ----
    for fn in (one_batch, new, do_item, show_results, show_batch):
        _attach_method(dl, fn)

    # ---- attach vocab if available ----
    if hasattr(dl.dataset, "vocab"):
        vocab = dl.dataset.vocab
        if isinstance(vocab, list):
            from fastai.data.transforms import CategoryMap
            vocab = CategoryMap(vocab)
        dl.vocab = vocab

    # ---- attach x/y keys if available ----
    if hasattr(dl.dataset, "x_keys"):
        dl.x_keys = dl.dataset.x_keys

    if hasattr(dl.dataset, "y_keys"):
        dl.y_keys = dl.dataset.y_keys

    return dl

In [ ]:
#| hide
from fastcore.test import test_eq
from torch.utils.data import DataLoader

def test_patch_dataloader():
    class DummyDS:
        def __init__(self):
            self.data = [([1, 2], 0), ([3, 4], 1)]
            self.vocab = ["cat", "dog"]
            self.x_keys = ["image"]
            self.y_keys = ["label"]
        def __len__(self): return len(self.data)
        def __getitem__(self, i): return self.data[i]

    dl = DataLoader(DummyDS(), batch_size=2)
    patched = _patch_dataloader(dl)

    # Verify methods attached
    test_eq(hasattr(patched, 'one_batch'), True)
    test_eq(hasattr(patched, 'new'), True)
    test_eq(hasattr(patched, 'show_batch'), True)

    # Verify attributes attached
    test_eq(list(patched.vocab), ["cat", "dog"])
    test_eq(patched.x_keys, ["image"])
    
    # Test method functionality
    batch = patched.one_batch()
    test_eq(len(batch), 2)
    
    # Test clone
    cloned = patched.new(batch_size=1)
    test_eq(cloned.batch_size, 1)
    test_eq(hasattr(cloned, 'one_batch'), True)

    return "_patch_dataloader tests passed"

test_eq(test_patch_dataloader(), "_patch_dataloader tests passed")

In [ ]:
#| export
def _show_summary(train_dl, val_dl=None):
    """
    Print a summary of the training and validation dataloaders.

    Displays dataset size, batch size, number of batches,
    batch shapes/dtypes, and approximate memory usage (MB).
    """
    def _describe_dl(dl, name):
        print(f"\n{name} DataLoader")
        print("-" * (len(name) + 11))

        ds = dl.dataset

        # dataset info
        try:
            ds_len = len(ds)
        except:
            ds_len = "unknown"

        print(f"Dataset size : {ds_len}")
        print(f"Batch size   : {dl.batch_size}")
        print(f"Batches      : {len(dl)}")

        if hasattr(ds, "vocab"):
            print(f"Classes      : {ds.vocab}")

        # inspect one batch
        try:
            batch = next(iter(dl))
        except Exception as e:
            print(f"Could not fetch batch: {e}")
            return

        print("\nBatch structure:")

        def _tensor_size(x):
            return x.numel() * x.element_size() / (1024 ** 2)  # MB

        total_mem = 0.0

        if isinstance(batch, (list, tuple)):
            for i, item in enumerate(batch):
                if isinstance(item, torchTensor):
                    mem = _tensor_size(item)
                    total_mem += mem
                    print(f"  [{i}] shape={tuple(item.shape)} dtype={item.dtype} ~{mem:.2f} MB")
                else:
                    print(f"  [{i}] type={type(item)}")
        elif isinstance(batch, dict):
            for k, v in batch.items():
                if isinstance(v, torchTensor):
                    mem = _tensor_size(v)
                    total_mem += mem
                    print(f"  {k}: shape={tuple(v.shape)} dtype={v.dtype} ~{mem:.2f} MB")
                else:
                    print(f"  {k}: type={type(v)}")
        else:
            print(type(batch))

        print(f"Approx batch memory: {total_mem:.2f} MB")

    _describe_dl(train_dl, "Train")

    if val_dl is not None:
        _describe_dl(val_dl, "Valid")

In [ ]:
#| hide
import torch
from unittest.mock import patch
from torch.utils.data import TensorDataset, DataLoader
from fastcore.test import test_eq

def test_show_summary():
    # Dummy dataloaders
    dummy_x = torch.zeros(4, 1, 10, 10)
    dummy_y = torch.ones(4, dtype=torch.long)
    dummy_ds = TensorDataset(dummy_x, dummy_y)
    train_dl = DataLoader(dummy_ds, batch_size=2)
    val_dl = DataLoader(dummy_ds, batch_size=2)

    # Capture print output
    with patch('builtins.print') as mock_print:
        _show_summary(train_dl, val_dl)
        output = "\n".join(str(call.args[0]) for call in mock_print.call_args_list if call.args)

    # Verify output contents
    test_eq("Train DataLoader" in output, True)
    test_eq("Valid DataLoader" in output, True)
    test_eq("Batch size   : 2" in output, True)
    test_eq("Batches      : 2" in output, True)

    return "_show_summary tests passed"

test_eq(test_show_summary(), "_show_summary tests passed")

### Pipeline Context

**`PipelineContext`** is a central data structure (implemented as a standard Python `@dataclass`) designed to hold the entire state of the data loading pipeline as it progresses. It efficiently stores raw inputs, task configurations, intermediate dataset objects, and the final constructed DataLoaders (`dls`). This unified state object makes it easy to pass context across different stages of the bioMONAI architecture.

In [ ]:
#| export
@dataclass
class PipelineContext:

    # raw inputs
    data: Any = None
    val_data: Any = None

    # task/config
    task: Optional[str] = None
    dataset_name: Optional[str] = None
    backend: Optional[str] = None
    mode: str = "train"

    # loaded/intermediate state
    records: Optional[list[dict]] = None

    train_ds: Any = None
    valid_ds: Any = None

    dls: Any = None

    # unified config
    config: dict = field(default_factory=dict)

    # runtime metadata
    metadata: dict = field(default_factory=dict)

In [ ]:
# Example: Initializing and updating the state within a PipelineContext

# 1. Create a context with raw data and a task configuration
ctx = PipelineContext(
    data=["scan1.nii.gz", "scan2.nii.gz"],
    task="segmentation",
    backend="pytorch"
)

# 2. Simulate pipeline progression by storing intermediate state/config
ctx.metadata["num_classes"] = 3
ctx.config["batch_size"] = 16
ctx.train_ds = "DatasetObjectPlaceholder"

print(f"Task:    {ctx.task}")
print(f"Backend: {ctx.backend}")
print(f"Data:    {ctx.data}")
print(f"Config:  {ctx.config}")

Task:    segmentation
Backend: pytorch
Data:    ['scan1.nii.gz', 'scan2.nii.gz']
Config:  {'batch_size': 16}


In [ ]:
#| hide
from fastcore.test import test_eq

def test_pipeline_context():
    # Initialize
    ctx = PipelineContext(
        data=[1, 2, 3],
        task="classification",
        mode="train"
    )
    
    # Modify state
    ctx.config["lr"] = 1e-3
    ctx.metadata["info"] = "test"
    ctx.train_ds = "dummy_ds"

    # Verify
    test_eq(ctx.data, [1, 2, 3])
    test_eq(ctx.task, "classification")
    test_eq(ctx.mode, "train")
    test_eq(ctx.config["lr"], 1e-3)
    test_eq(ctx.metadata["info"], "test")
    test_eq(ctx.train_ds, "dummy_ds")

    return "PipelineContext tests passed"

test_eq(test_pipeline_context(), "PipelineContext tests passed")

### Source Detection & Factory

The library provides an intelligent, automated routing mechanism to handle diverse raw data formats seamlessly.

**`detect_source`** analyzes the structure of the input data (e.g., Pandas DataFrames, CSV file paths, directories, lists, or dictionaries) and identifies the underlying data type. This allows the system to determine the appropriate `Source` handler without requiring explicit user configuration.

**`build_source`** acts as the dynamic factory for your data pipeline. It leverages `detect_source` to identify the data structure and automatically instantiates the corresponding handler class from the `SOURCE_REGISTRY`. This enables you to pass raw data directly into the pipeline without manually importing or configuring specific Source classes.

In [ ]:
#| export
def detect_source(data):

    if callable(data):
        return "callable"

    if isinstance(data, pd.DataFrame):
        return "dataframe"

    # single dict
    if isinstance(data, dict):
        return "dict"

    # list/tuple handling
    if isinstance(data, (list, tuple)):

        if len(data) == 0:
            return "list"

        # list of dicts
        if all(isinstance(x, dict) for x in data):
            return "dict"

        return "list"

    if isinstance(data, str):

        if data.endswith(".csv"):
            return "csv"

        if Path(data).is_dir():
            return "folder"

    raise ValueError("Cannot detect data source")

In [ ]:
# Example: Automatically detecting data source types
import pandas as pd

# Define various data formats
sample_csv = "dataset_metadata.csv"
sample_df = pd.DataFrame({"image": ["img1.png", "img2.png"], "label": [0, 1]})
sample_list = ["train_images", "val_images"]

# Detect sources
print(f"CSV string: {detect_source(sample_csv)}")
print(f"DataFrame:  {detect_source(sample_df)}")
print(f"List:       {detect_source(sample_list)}")

CSV string: csv
DataFrame:  dataframe
List:       list


In [ ]:
#| hide
import pandas as pd
from fastcore.test import test_eq

def test_detect_source():
    # DataFrame
    df = pd.DataFrame({"img": ["i1", "i2"]})
    test_eq(detect_source(df), 'dataframe')

    # List
    test_eq(detect_source(["f1", "f2"]), 'list')
    
    # List of Dicts
    test_eq(detect_source([{"a": 1}, {"b": 2}]), 'dict')

    # CSV string
    test_eq(detect_source("data.csv"), 'csv')
    
    # Folder path
    test_eq(detect_source('.'), 'folder')

    return "detect_source tests passed"

test_eq(test_detect_source(), "detect_source tests passed")

In [ ]:
#| export
def build_source(data, **kwargs):

    name = detect_source(data)

    source_cls = SOURCE_REGISTRY[name]

    return source_cls(data, **kwargs)

In [ ]:
# Example: Using build_source as an automatic factory

# 1. Register a custom handler for "list" data types
@register_source("list")
class DemoListSource:
    def __init__(self, data, **kwargs):
        self.data = data
        self.kwargs = kwargs

# 2. Provide raw data
raw_data = ["scan_01.nii.gz", "scan_02.nii.gz"]

# 3. Automatically detect and build the appropriate source
my_source = build_source(raw_data, base_path="/medical_data")

print(f"Built type: {type(my_source).__name__}")
print(f"Kwargs:     {my_source.kwargs}")

Built type: DemoListSource
Kwargs:     {'base_path': '/medical_data'}


In [ ]:
#| hide
from fastcore.test import test_eq

def test_build_source():
    # Backup registry
    orig_registry = SOURCE_REGISTRY.copy()

    # Register dummy class
    class DummyDictSource:
        def __init__(self, data, **kwargs):
            self.data = data
            self.kwargs = kwargs

    SOURCE_REGISTRY["dict"] = DummyDictSource

    # Test factory routing
    test_dict = {"image": "img.png", "label": 1}
    built = build_source(test_dict, extra_arg=True)

    # Verify instantiation and kwargs
    test_eq(isinstance(built, DummyDictSource), True)
    test_eq(built.data, test_dict)
    test_eq(built.kwargs, {"extra_arg": True})

    # Cleanup
    SOURCE_REGISTRY.clear()
    SOURCE_REGISTRY.update(orig_registry)

    return "build_source tests passed"

test_eq(test_build_source(), "build_source tests passed")

### Data Sources

The library employs a hierarchical system of **Source** classes designed to ingest and standardize various raw data formats seamlessly. All specialized handlers ultimately convert their inputs into records and delegate the heavy lifting to the core `DictSource` engine, ensuring a consistent processing pipeline regardless of the original data structure.

* **`BaseSource`:** The foundational abstract class. It provides essential built-in utilities for parsing raw data, mapping dictionary keys, handling missing values, and dynamically resolving file paths (applying base paths, folders, and suffixes).
* **`DictSource`:** The core processing engine. It processes single dictionaries or lists of dictionaries, featuring robust key mapping capabilities, automatic path formatting, dynamic value generation via callables, and seamless integration with class-specific loaders.
* **`DataFrameSource`:** A specialized adapter for Pandas DataFrames. It converts tabular rows into dictionary records, enabling you to leverage all of `DictSource`'s features directly on tabular data.
* **`CSVSource`:** A highly convenient wrapper that automatically reads a CSV file into a Pandas DataFrame and delegates it to `DataFrameSource`.
* **`FolderSource`:** Automates the ingestion of data directly from directory structures. It scans specified folders, sorts contents, and automatically pairs corresponding files (e.g., raw images with their respective masks) into unified records.
* **`ListSource`:** Gracefully handles standard Python lists. It accepts either flat lists of file paths (automatically mapped to an `image` key) or lists of dictionaries, routing them through the dictionary processing engine.
* **`CallableSource`:** Enables dynamic, on-the-fly dataset generation using Python functions. This is ideal when data must be retrieved from a database, an API, or derived via custom logic during runtime.

In [ ]:
#| export
class BaseSource:
    def __init__(
            self, 
            data, 
            x_keys=None,
            y_keys=None,
            x_class=None,
            y_class=None,
            get_items=None,
            base_path=None,
            folders=None,
            suffixes=None,
            keep_original=False,
            **kwargs):
        
        store_attr()
        self.kwargs = kwargs

    def load(self):
        raise NotImplementedError

    def df(self):
        return pd.DataFrame(self.load())
    
    def peak(self):
        return self.df().head()
    
    def _add_key_mappings(self, keys):
        if keys is None:
            return
        if isinstance(keys, str):
            keys = [keys]
        for key in keys:
            self.get_items.setdefault(key, key)

    def _build_class_loaders(self):
        loaders = []
        for cls, keys in ((self.x_class, self.x_keys), (self.y_class, self.y_keys)):
            if cls and hasattr(cls, "get_dict_label"):
                loader_kwargs = dict(self.kwargs)
                if keys is not None:
                    loader_kwargs["keys"] = keys
                loaders.append(cls.get_dict_label(**loader_kwargs))
        return loaders

    def _has_path_config(self, new_col):
        return (
            new_col in self.folders
            or new_col in self.suffixes
            or self.base_path != Path()
        )

    def _is_missing(self, value):
        if value is None:
            return True
        if isinstance(value, (list, tuple, dict)):
            return False
        try:
            return bool(pd.isna(value))
        except (TypeError, ValueError):
            return False

    def _build_path(self, value, new_col):
        if self._is_missing(value):
            return None

        value = Path(str(value))
        suffix = self.suffixes.get(new_col, "")
        if suffix and not value.name.endswith(suffix):
            value = value.with_name(f"{value.name}{suffix}")
        if value.is_absolute():
            return value.as_posix()

        base = self.base_path

        folder = self.folders.get(new_col)
        if folder:
            base = base / folder

        return (base / value).as_posix()

    def _build_paths(self, values, new_col):
        return [self._build_path(v, new_col) for v in values]

    def _format_value(self, value, new_col):
        if not self._has_path_config(new_col):
            return value
        if isinstance(value, (list, tuple)):
            return self._build_paths(value, new_col)
        return self._build_path(value, new_col)

In [ ]:
# Example: Creating a custom data source inheriting from BaseSource
import pandas as pd
from pathlib import Path

class MyCustomSource(BaseSource):
    def load(self):
        raw_data = [{"image": "scan_01"}, {"image": "scan_02"}]
        # Automatically apply base paths, folders, and suffixes
        for row in raw_data:
            row["image"] = self._format_value(row["image"], new_col="image")
        return raw_data

# Initialize with specific path formatting rules
my_source = MyCustomSource(
    data=None, 
    base_path=Path("/medical_data"),
    folders={"image": "images"},
    suffixes={"image": ".nii.gz"}
)

# View the generated DataFrame
print(my_source.peak())

                                 image
0  /medical_data/images/scan_01.nii.gz
1  /medical_data/images/scan_02.nii.gz


In [ ]:
#| hide
import pandas as pd
from pathlib import Path
from fastcore.test import test_eq

def test_basesource():
    base_src = BaseSource(
        data=None,
        base_path=Path("dataset"),
        folders={"image": "scans"},
        suffixes={"image": ".nii.gz"},
        get_items={}
    )

    # Missing values
    test_eq(base_src._is_missing(None), True)
    test_eq(base_src._is_missing(pd.NA), True)
    test_eq(base_src._is_missing("valid_string"), False)

    # Path configs
    test_eq(base_src._has_path_config("image"), True)

    # Path building
    test_eq(base_src._build_path("patient_01", "image"), "dataset/scans/patient_01.nii.gz")
    test_eq(base_src._build_path("patient_02.nii.gz", "image"), "dataset/scans/patient_02.nii.gz")

    # Format list of values
    test_eq(
        base_src._format_value(["pt1", "pt2"], "image"), 
        ["dataset/scans/pt1.nii.gz", "dataset/scans/pt2.nii.gz"]
    )

    # Key mappings
    base_src._add_key_mappings(["mask", "label"])
    test_eq(base_src.get_items["mask"], "mask")
    test_eq(base_src.get_items["label"], "label")

    return "BaseSource tests passed"

test_eq(test_basesource(), "BaseSource tests passed")

In [ ]:
#| export
@register_source("dict")
class DictSource(BaseSource):
    def __init__(
        self,
        data,
        x_keys=None,
        y_keys=None,
        x_class=None,
        y_class=None,
        get_items=None,
        base_path=None,
        folders=None,
        suffixes=None,
        keep_original=False,
        **kwargs,
    ):
        if isinstance(data, dict):
            data = [data]

        self.data = list(data)

        self.get_items = dict(get_items or {})
        self._add_key_mappings(x_keys)
        self._add_key_mappings(y_keys)

        self.base_path = Path(base_path) if base_path else Path()
        self.folders = folders or {}
        self.suffixes = suffixes or {}
        self.keep_original = keep_original
        self.x_keys = x_keys
        self.y_keys = y_keys
        self.x_class = x_class
        self.y_class = y_class
        self.kwargs = kwargs

        self.loaders = self._build_class_loaders()

    def _resolve(self):
        out = []

        for row in self.data:

            item = dict(row) if self.keep_original else {}

            for new_col, src_col in self.get_items.items():

                if isinstance(src_col, str):
                    value = row.get(src_col)
                    item[new_col] = self._format_value(value, new_col)

                elif isinstance(src_col, (list, tuple)):
                    values = [row.get(c) for c in src_col]
                    item[new_col] = self._format_value(values, new_col)

                elif isinstance(src_col, Callable):
                    value = src_col(row)
                    item[new_col] = self._format_value(value, new_col)

                else:
                    raise ValueError(
                        f"Invalid src_col type for {new_col}: {type(src_col)}"
                    )

            for loader in self.loaders:
                item = loader(item)

            out.append(item)

        return out

    def load(self):
        return self._resolve()

In [ ]:
# Example: Loading and preprocessing data from a dictionary

raw_dict_data = {
    "ch1": "img001_ch1.nii.gz", 
    "ch2": "img001_ch2", 
    "label": "A", 
    "scale": 2
}

# Map keys, combine channels, apply functions, and format paths
dict_source = DictSource(
    data=raw_dict_data,
    y_keys="label",
    y_class=BioLabel, # Automatically encodes text to int based on vocab
    get_items={
        "image": ("ch1", "ch2"), 
        "weight": lambda row: row["scale"] * 10
    },
    folders={"image": "data/images"},
    suffixes={"image": ".nii.gz"},
    vocab=['A', 'B']
)

print(dict_source.load())

[{'image': ['data/images/img001_ch1.nii.gz', 'data/images/img001_ch2.nii.gz'], 'weight': 20, 'label': TensorCategory(0)}]


In [ ]:
#| hide
from fastcore.test import test_eq

def test_dictsource():
    source = DictSource(
        {"ch1": "img001_ch1.nii.gz", "ch2": "img001_ch2", "label": "A", "scale": 2},
        y_keys="label",
        y_class=BioLabel,
        get_items={"image": ("ch1", "ch2"), "weight": lambda row: row["scale"] * 10},
        folders={"image": "data/images"},
        suffixes={"image": ".nii.gz"},
        vocab=['A', 'B'],
    )

    # Verify mappings, path formatting, and loaders
    res = source.load()
    test_eq(res, [{
        "image": ["data/images/img001_ch1.nii.gz", "data/images/img001_ch2.nii.gz"],
        "weight": 20,
        "label": 0,
    }])

    return "DictSource tests passed"

test_eq(test_dictsource(), "DictSource tests passed")

In [ ]:
#| export
@register_source("dataframe")
class DataFrameSource(BaseSource):
    def __init__(
        self,
        df,
        x_keys=None,
        y_keys=None,
        get_items=None,
        base_path=None,
        folders=None,
        suffixes=None,
        keep_original=False,
        **kwargs,
    ):

        self.source = DictSource(
            data=df.to_dict(orient="records"),
            x_keys=x_keys,
            y_keys=y_keys,
            get_items=get_items,
            base_path=base_path,
            folders=folders,
            suffixes=suffixes,
            keep_original=keep_original,
            **kwargs
        )

    def load(self):
        return self.source.load()
    

In [ ]:
import pandas as pd

# Example 1: Combining multiple columns into a single channel list
df_multi = pd.DataFrame({
    "channel1": ["img001_ch1", "img002_ch1"],
    "channel2": ["img001_ch2", "img002_ch2"],
    "mask": ["mask001", "mask002"]
})

source_multi = DataFrameSource(
    df=df_multi,
    get_items={"image": ["channel1", "channel2"], "label": "mask"},
    base_path="data",
    folders={"image": "images", "label": "masks"},
    suffixes={"image": ".png", "label": ".nii.gz"}
)

print("--- Example 1: Multi-channel Output ---")
for item in source_multi.load():
    print(item)

--- Example 1: Multi-channel Output ---
{'image': ['data/images/img001_ch1.png', 'data/images/img001_ch2.png'], 'label': 'data/masks/mask001.nii.gz'}
{'image': ['data/images/img002_ch1.png', 'data/images/img002_ch2.png'], 'label': 'data/masks/mask002.nii.gz'}


In [ ]:
# Example 2: Single column mapping and exporting back to a clean DataFrame
df_single = pd.DataFrame({
    "filename": ["img001", "img002"],
    "labels": [0, 1]
})

source_single = DataFrameSource(
    df=df_single,
    y_keys="labels",
    get_items={"image": "filename"},
    folders={"image": "data/images"},
    suffixes={"image": ".nii.gz"}
)

print("\n--- Example 2: Exported as Pandas DataFrame ---")
print(source_single.df())


--- Example 2: Exported as Pandas DataFrame ---
                       image  labels
0  data/images/img001.nii.gz       0
1  data/images/img002.nii.gz       1


In [ ]:
# Example 3: Keep the original DataFrame data without path modifications
source_asis = DataFrameSource(
    df=df_single,
    get_items=None,
    keep_original=True
)

print("\n--- Example 3: Kept Original Data ---")
print(source_asis.load())


--- Example 3: Kept Original Data ---
[{'filename': 'img001', 'labels': 0}, {'filename': 'img002', 'labels': 1}]


In [ ]:
#| hide
import pandas as pd
from fastcore.test import test_eq

def test_dataframesource():
    # Setup
    df_multi = pd.DataFrame({
        "channel1": ["img001_ch1", "img002_ch1"],
        "channel2": ["img001_ch2", "img002_ch2"],
        "mask": ["mask001", "mask002"]
    })
    
    df_single = pd.DataFrame({
        "filename": ["img001", "img002"],
        "labels": [0, 1]
    })

    # Test 1: Multi-channel
    source_multi = DataFrameSource(
        df=df_multi,
        get_items={"image": ["channel1", "channel2"], "label": "mask"},
        base_path="data",
        folders={"image": "images", "label": "masks"},
        suffixes={"image": ".png", "label": ".nii.gz"}
    )
    expected_multi = [
        {'image': ['data/images/img001_ch1.png', 'data/images/img001_ch2.png'], 'label': 'data/masks/mask001.nii.gz'},
        {'image': ['data/images/img002_ch1.png', 'data/images/img002_ch2.png'], 'label': 'data/masks/mask002.nii.gz'}
    ]
    test_eq(source_multi.load(), expected_multi)

    # Test 2: Single column & DataFrame generation
    source_single = DataFrameSource(
        df=df_single,
        y_keys="labels",
        get_items={"image": "filename"},
        folders={"image": "data/images"},
        suffixes={"image": ".nii.gz"}
    )
    expected_df_single = pd.DataFrame({
        "labels": [0, 1],
        "image": ["data/images/img001.nii.gz", "data/images/img002.nii.gz"]
    })
    test_eq(source_single.df().to_dict(orient="records"), expected_df_single.to_dict(orient="records"))

    # Test 3: Keep original
    source_asis = DataFrameSource(df=df_single, get_items=None, keep_original=True)
    test_eq(source_asis.load(), df_single.to_dict(orient="records"))

    # Test 4: String labels mapping
    df_str = pd.DataFrame({"filename": ["img001", "img002"], "labels": ["catA", "catB"]})
    source_str = DataFrameSource(
        df=df_str, y_keys="labels", get_items={"image": "filename"}, 
        folders={"image": "data/images"}, suffixes={"image": ".nii.gz"}
    )
    expected_str = [
        {'labels': 'catA', 'image': 'data/images/img001.nii.gz'},
        {'labels': 'catB', 'image': 'data/images/img002.nii.gz'}
    ]
    test_eq(source_str.load(), expected_str)

    return "DataFrameSource tests passed"

test_eq(test_dataframesource(), "DataFrameSource tests passed")

In [ ]:
#| export
@register_source("csv")
class CSVSource(BaseSource):

    @delegates(DataFrameSource.__init__)
    def __init__(self, path, **kwargs):
        self.path = path

        df = pd.read_csv(path)
        source = DataFrameSource(df, **kwargs)
        self.source = source

    def load(self):
        return self.source.load()

In [ ]:
import pandas as pd
from pathlib import Path

# Example: Loading and mapping data directly from a CSV file

# 1. Create a dummy CSV file
sample_csv = Path("sample_dataset.csv")
pd.DataFrame({
    "image_file": ["scan_001", "scan_002"],
    "diagnosis": ["Healthy", "Pathological"]
}).to_csv(sample_csv, index=False)

# 2. Configure CSVSource to read and format the data
csv_source = CSVSource(
    path=sample_csv,
    y_keys="diagnosis",
    get_items={"image": "image_file"},
    folders={"image": "medical_scans"},
    suffixes={"image": ".nii.gz"}
)

print("Processed Output:")
for item in csv_source.load():
    print(item)

# Cleanup
if sample_csv.exists():
    sample_csv.unlink()

Processed Output:
{'image': 'medical_scans/scan_001.nii.gz', 'diagnosis': 'Healthy'}
{'image': 'medical_scans/scan_002.nii.gz', 'diagnosis': 'Pathological'}


In [ ]:
#| hide
import pandas as pd
from pathlib import Path
from fastcore.test import test_eq

def test_csvsource():
    test_csv = Path("test_dummy_data.csv")
    pd.DataFrame({
        "ch1": ["img1_c1", "img2_c1"],
        "ch2": ["img1_c2", "img2_c2"],
        "mask": ["m1", "m2"]
    }).to_csv(test_csv, index=False)

    source = CSVSource(
        path=test_csv,
        get_items={"image": ["ch1", "ch2"], "label": "mask"},
        base_path="data",
        folders={"image": "images", "label": "masks"},
        suffixes={"image": ".png", "label": ".nii.gz"}
    )

    expected = [
        {'image': ['data/images/img1_c1.png', 'data/images/img1_c2.png'], 'label': 'data/masks/m1.nii.gz'},
        {'image': ['data/images/img2_c1.png', 'data/images/img2_c2.png'], 'label': 'data/masks/m2.nii.gz'}
    ]
    
    test_eq(source.load(), expected)

    if test_csv.exists():
        test_csv.unlink()

    return "CSVSource tests passed"

test_eq(test_csvsource(), "CSVSource tests passed")

In [ ]:
#| export
@register_source("folder")
class FolderSource(BaseSource):

    def __init__(self, root, get_items, **kwargs):
        self.root = Path(root)
        self.get_items = get_items

        scanned = {
            key: self._scan(folder)
            for key, folder in self.get_items.items()
        }

        records = [
            dict(zip(scanned.keys(), values))
            for values in zip(*scanned.values())
        ]

        self.source = DictSource(
            records,
            get_items={k: k for k in scanned.keys()},
            base_path=self.root,
            folders=self.get_items,
            keep_original=True,
            **kwargs,
        )

    def _scan(self, folder):

        path = self.root / folder
        return sorted([f.name for f in path.iterdir()])

    def load(self):
        return self.source.load()

In [ ]:
import tempfile
from pathlib import Path

# Example: Automatically pairing images and masks from directories

# 1. Create a temporary dummy dataset structure
with tempfile.TemporaryDirectory() as tmpdir:
    root = Path(tmpdir)
    (root / "images").mkdir()
    (root / "masks").mkdir()
    
    (root / "images" / "scan_01.nii.gz").touch()
    (root / "images" / "scan_02.nii.gz").touch()
    (root / "masks" / "scan_01.nii.gz").touch()
    (root / "masks" / "scan_02.nii.gz").touch()

    # 2. Configure FolderSource to scan and pair
    folder_source = FolderSource(
        root=root,
        get_items={"image": "images", "label": "masks"}
    )

    print("Matched Pairs:")
    for item in folder_source.load():
        print(item)

Matched Pairs:
{'image': '/tmp/tmpj40kdmc5/images/scan_01.nii.gz', 'label': '/tmp/tmpj40kdmc5/masks/scan_01.nii.gz'}
{'image': '/tmp/tmpj40kdmc5/images/scan_02.nii.gz', 'label': '/tmp/tmpj40kdmc5/masks/scan_02.nii.gz'}


In [ ]:
#| hide
import tempfile
from pathlib import Path
from fastcore.test import test_eq

def test_foldersource():
    with tempfile.TemporaryDirectory() as tmp:
        root = Path(tmp)
        (root / "images").mkdir()
        (root / "masks").mkdir()

        (root / "images" / "img001.nii.gz").touch()
        (root / "images" / "img002.nii.gz").touch()
        (root / "masks" / "img001.nii.gz").touch()
        (root / "masks" / "img002.nii.gz").touch()

        source = FolderSource(
            root,
            get_items={"image": "images", "label": "masks"}
        )

        res = source.load()
        
        expected = [
            {"image": str(root / "images" / "img001.nii.gz"), "label": str(root / "masks" / "img001.nii.gz")},
            {"image": str(root / "images" / "img002.nii.gz"), "label": str(root / "masks" / "img002.nii.gz")}
        ]

        test_eq(res, expected)

    return "FolderSource tests passed"

test_eq(test_foldersource(), "FolderSource tests passed")

In [ ]:
#| export
@register_source("list")
class ListSource(BaseSource):

    def __init__(
        self,
        items,
        get_items=None,
        base_path=".",
        keep_original=True,
        **kwargs,
    ):
        self.items = items
        self.get_items = get_items
        self.base_path = base_path
        self.keep_original = keep_original
        self.kwargs = kwargs

    def load(self):

        if not self.items:
            raise ValueError("`items` cannot be empty.")

        first = self.items[0]

        # List of paths
        if isinstance(first, (str, Path)):
            records = [{"image": str(x)} for x in self.items]
            get_items = {"image": "image"}

        # List of dict-like records
        elif isinstance(first, dict):
            records = [dict(x) for x in self.items]
            get_items = self.get_items

        else:
            raise TypeError(
                "`items` must be a list of paths or dictionaries."
            )

        return DictSource(
            records,
            get_items=get_items,
            base_path=self.base_path,
            keep_original=self.keep_original,
            **self.kwargs,
        ).load()

In [ ]:
# Example: Loading and mapping data from a list of dictionaries

raw_list = [
    {"img": "scan_001.nii.gz", "mask": "seg_001.nii.gz"},
    {"img": "scan_002.nii.gz", "mask": "seg_002.nii.gz"}
]

# Map keys and apply a base directory path
list_source = ListSource(
    items=raw_list,
    get_items={"image": "img", "label": "mask"},
    base_path="medical_data",
    keep_original=False
)

for item in list_source.load():
    print(item)

{'image': 'medical_data/scan_001.nii.gz', 'label': 'medical_data/seg_001.nii.gz'}
{'image': 'medical_data/scan_002.nii.gz', 'label': 'medical_data/seg_002.nii.gz'}


In [ ]:
#| hide
import pandas as pd
from fastcore.test import test_eq

def test_listsource():
    # Test list of dicts
    items = [{"img": "img1", "mask": "mask1"}, {"img": "img2", "mask": "mask2"}]
    source_dict = ListSource(
        items,
        get_items={"image": "img", "label": "mask"},
        base_path="data",
        keep_original=False,
    )
    expected_dict = [
        {"image": "data/img1", "label": "data/mask1"},
        {"image": "data/img2", "label": "data/mask2"}
    ]
    test_eq(source_dict.load(), expected_dict)

    # Test flat list of paths
    flat_list = ["scan1.png", "scan2.png"]
    source_flat = ListSource(flat_list, base_path="data")
    expected_flat = [
        {"image": "data/scan1.png"},
        {"image": "data/scan2.png"}
    ]
    test_eq(source_flat.load(), expected_flat)

    return "ListSource tests passed"

test_eq(test_listsource(), "ListSource tests passed")

In [ ]:
#| export
@register_source("callable")
class CallableSource(BaseSource):

    @delegates(DictSource.__init__)
    def __init__(
        self,
        items_fn,
        target_fn=None,
        x_keys="image",
        y_keys="label",
        **kwargs,
    ):
        self.items_fn = items_fn
        self.target_fn = target_fn
        self.x_keys = x_keys
        self.y_keys = y_keys
        self.kwargs = kwargs

    def load(self):

        items = list(self.items_fn())

        first = items[0]

        # Case 1: items already dictionaries
        if isinstance(first, dict):

            records = items
            if self.target_fn:
                 records = [dict(row, **{self.y_keys: self.target_fn(row[self.x_keys])}) for row in records]

        # Case 2: items are inputs only (paths, ids, etc.)
        else:

            records = []

            for x in items:

                row = {
                    self.x_keys: x
                }

                if self.target_fn:
                    row[self.y_keys] = self.target_fn(x)

                records.append(row)

        source = DictSource(
            records,
            x_keys=self.x_keys,
            y_keys=self.y_keys,
            **self.kwargs,
        )

        return source.load()

In [ ]:
# Example 1: Creating a dataset dynamically using functions (Callables)
# This is highly useful when your data requires custom logic, database queries, or API calls.

def get_image_ids():
    # Simulating a function that retrieves a list of IDs or paths
    return ["img1", "img2"]

def generate_mask_path(image_id):
    # Simulating a logic that derives the target mask path from the image ID
    return image_id.replace("img", "mask")

# Configure CallableSource to map these functions
source_from_strings = CallableSource(
    items_fn=get_image_ids, 
    target_fn=generate_mask_path, 
    x_keys="image", 
    y_keys="label"
)

print("--- Example 1: Callables returning raw strings ---")
for item in source_from_strings.load():
    print(item)

--- Example 1: Callables returning raw strings ---
{'image': 'img1', 'label': 'mask1'}
{'image': 'img2', 'label': 'mask2'}


In [ ]:
# Example 2: Using CallableSource when the generator returns dictionaries directly

def get_metadata_dicts():
    # Simulating a function that returns pre-structured records
    return [
        {'image': 'img1'},
        {'image': 'img2'}
    ]

def derive_label(img_path):
    # The target function operates on the specific key defined by x_keys ("image")
    return img_path.replace("img", "mask")

source_from_dicts = CallableSource(
    items_fn=get_metadata_dicts, 
    target_fn=derive_label, 
    x_keys="image", 
    y_keys="label"
)

print("\n--- Example 2: Callables returning dictionaries ---")
for item in source_from_dicts.load():
    print(item)


--- Example 2: Callables returning dictionaries ---
{'image': 'img1', 'label': 'mask1'}
{'image': 'img2', 'label': 'mask2'}


In [ ]:
#| hide
from fastcore.test import test_eq

def test_callablesource():
    def get_tgt(x): return x.replace("img", "mask")
    expected = [{'image': 'img1', 'label': 'mask1'}, {'image': 'img2', 'label': 'mask2'}]

    # Test lists of strings
    def get_strs(): return ["img1", "img2"]
    src_str = CallableSource(get_strs, get_tgt, x_keys="image", y_keys="label")
    test_eq(src_str.load(), expected)

    # Test lists of dictionaries
    def get_dicts(): return [{'image': 'img1'}, {'image': 'img2'}]
    src_dict = CallableSource(get_dicts, get_tgt, x_keys="image", y_keys="label")
    test_eq(src_dict.load(), expected)

    return "CallableSource tests passed"

test_eq(test_callablesource(), "CallableSource tests passed")

### Datasets & Pipeline Builders

This section comprises the components responsible for data partitioning, transformation management (both augmented and deterministic), and the final construction of *Dataset* objects ready to be consumed by DataLoaders. It supports multiple execution environments through a unified interface.

#### Architecture Mixins (Shared Logic)

* **`DataSplitMixin`:** Centralizes the automated logic for splitting the dataset into training and validation subsets. It dynamically analyzes the structure of incoming records to resolve and apply the optimal splitting strategy: column-based (`ColSplitter` if the `is_valid` key is detected), explicit name-based (`NameSplitter` if `split_name` is found), or standard random splitting (`TrainTestSplitter`) as a fallback method.
* **`MonaiTransformMixin`:** Provides advanced utilities to orchestrate MONAI transformation pipelines. It automatically composes loading and processing operations, detects which transformations contain random components (by inspecting properties like `prob` or `Randomizable` interfaces), and automatically generates an equivalent deterministic pipeline for the validation set, systematically removing or replacing data augmentations.

#### Dataset Builders (Backend Structurers)

* **`DataBlockBuilder` (Backend: `fastai`):** A specialized generator for fastai's `DataBlock` API (registered under the `"datablock"` backend). It automates workflow creation by inferring input and target columns from standard naming conventions, linking the partitioning logic from `DataSplitMixin`, wrapping transformations into native `Pipeline` objects, and initializing specific blocks via their factory methods (`get_datablock`).
* **`MonaiDatasetBuilder` (Backend: `monai`):** Designed to instantiate native `monai.data.Dataset` objects. It inherits the combined partitioning and transformation management capabilities to ensure that the generated training and validation sets are perfectly compatible with MONAI's multidimensional operations and medical tensors.
* **`CacheDatasetBuilder` (Backend: `monai` with Acceleration):** Extends the construction of MONAI environments by using `monai.data.CacheDataset` to store preprocessed data directly in memory, drastically accelerating training time per epoch. It utilizes advanced prefixed argument mapping (`train_` and `val_`) to allow for clean, transparent, and asymmetric configurations of read threads (`num_workers`) or cache rates.

In [ ]:
#| export
class DataSplitMixin:
    """Shared logic for splitting datalists into index-based splits."""

    # --------------------------------------------------
    def _resolve_splitter(self, data):

        if self.splitter:
            return self.splitter

        if isinstance(data, pd.DataFrame):
            data = data.to_dict("records")

        if isinstance(data, list) and data and 'is_valid' in data[0]:
            return ColSplitter()
        
        if isinstance(data, list) and data and 'split_name' in data[0]:
            return NameSplitter()

        return TrainTestSplitter(
            test_fraction=self.valid_fraction,
            random_state=self.seed,
        )

    # --------------------------------------------------
    def _split_data(self, data, mode="train"):

        if mode == "test":
            return None, data

        splitter = self._resolve_splitter(data)

        train_idx, valid_idx = splitter(data)

        train = [data[i] for i in train_idx]
        valid = [data[i] for i in valid_idx]

        return train, valid

In [ ]:
# Example: Adding automatic data splitting to a custom class using DataSplitMixin

class CustomDataBuilder(DataSplitMixin):
    def __init__(self, data, valid_fraction=0.2):
        self.data = data
        self.valid_fraction = valid_fraction
        self.seed = 42
        self.splitter = None # Let the mixin auto-detect the best splitter
        
    def build(self):
        # The mixin provides the _split_data method automatically
        train_data, valid_data = self._split_data(self.data, mode="train")
        return train_data, valid_data

# 1. Provide data with explicit validation flags
raw_data = [
    {"image": "img1.png", "is_valid": False},
    {"image": "img2.png", "is_valid": False},
    {"image": "img3.png", "is_valid": True}
]

# 2. The mixin will automatically detect the 'is_valid' key and use ColSplitter
builder = CustomDataBuilder(raw_data)
train_set, valid_set = builder.build()

print(f"Training items: {len(train_set)}")
print(f"Validation items: {len(valid_set)}")

Training items: 2
Validation items: 1


In [ ]:
#| hide
import pandas as pd
from fastcore.test import test_eq

def test_datasplitmixin():
    class DummySplitterBase(DataSplitMixin):
        def __init__(self, splitter=None, valid_fraction=0.2, seed=42):
            self.splitter = splitter
            self.valid_fraction = valid_fraction
            self.seed = seed

    dummy_base = DummySplitterBase()

    # Test 1: Test mode bypassing
    train_none, test_data = dummy_base._split_data([1, 2, 3], mode="test")
    test_eq(train_none, None)
    test_eq(test_data, [1, 2, 3])

    # Test 2: Auto-detecting ColSplitter
    dummy_data_col = [{"is_valid": False}, {"is_valid": True}]
    resolved_col_splitter = dummy_base._resolve_splitter(dummy_data_col)
    test_eq(callable(resolved_col_splitter), True)

    # Test 3: Auto-detecting NameSplitter
    dummy_data_name = [{"split_name": "train"}, {"split_name": "val"}]
    resolved_name_splitter = dummy_base._resolve_splitter(dummy_data_name)
    test_eq(callable(resolved_name_splitter), True)

    # Test 4: Fallback to TrainTestSplitter
    dummy_data_basic = [{"image": "a"}, {"image": "b"}]
    resolved_fallback_splitter = dummy_base._resolve_splitter(dummy_data_basic)
    test_eq(callable(resolved_fallback_splitter), True)

    # Test 5: DataFrame handling
    dummy_df = pd.DataFrame({"colA": [1, 2]})
    resolved_df_splitter = dummy_base._resolve_splitter(dummy_df)
    test_eq(callable(resolved_df_splitter), True)

    return "DataSplitMixin tests passed"

test_eq(test_datasplitmixin(), "DataSplitMixin tests passed")

In [ ]:
#| export
class MonaiTransformMixin:
    """Shared MONAI transform helpers."""

    # --------------------------------------------------
    def _prepare_transform(self, transforms, loaders=None):

        if isinstance(transforms, Callable):
            transforms = [transforms]
        
        if loaders:
            if isinstance(loaders, Callable):
                loaders = [loaders]
            # put loaders as first transforms
            transforms = [*loaders, *(transforms or [])]

        if transforms is None:
            return None

        if isinstance(transforms, (list, tuple)):
            return Compose(transforms)
        return transforms

    # --------------------------------------------------
    # make this more general to detect any random transform, not just MONAI's
    def _is_random(self, t):
        is_rnd = (hasattr(t, "prob") or     
                #   isinstance(t, RandTransform) or
                #   isinstance(t, RandMonaiTransform) or
                  isinstance(t, Randomizable) or 
                  getattr(t, "_is_random", False))
        return is_rnd

    # --------------------------------------------------
    def _make_deterministic_transforms(self, transforms):
        """
        Convert random transforms into deterministic equivalents.
        """

        if transforms is None:
            return None

        # Normalize to list
        if isinstance(transforms, Compose):
            transforms = transforms.transforms

        deterministic = []

        for t in transforms:
            val_t = getattr(t, "_val_transform", None)

            # Replace random transforms with deterministic versions, when available
            if val_t:
                keys = getattr(t, "keys", None)
                kw = getattr(t, "det_kwargs", {})
                deterministic.append(val_t(keys=keys, **kw))
                continue

            # Skip other random transforms
            if self._is_random(t):
                continue

            deterministic.append(t)

        return Compose(deterministic)
    
    def _build_transforms_and_loaders(self):
        if self.val_item_transforms is None:
            self.val_item_transforms = self._make_deterministic_transforms(
                self.item_transforms
            )

        if self.get_x is not None:
            x_loader = self.get_x
        elif self.x_class and hasattr(self.x_class, "get_dict_loader"):
            x_loader = self.x_class.get_dict_loader(
                keys=self.x_keys,
                transforms=self.item_transforms,
                **self.kwargs,
            )
        else:
            x_loader = None

        if self.get_y is not None:
            y_loader = self.get_y
        elif self.y_class and hasattr(self.y_class, "get_dict_loader"):
            y_loader = self.y_class.get_dict_loader(
                keys=self.y_keys,
                transforms=self.val_item_transforms,
                **self.kwargs,
            )
        else:
            y_loader = None

        # Remove None loaders
        loaders = [l for l in [x_loader, y_loader] if l is not None]

        train_transform = self._prepare_transform(
            self.transforms,
            loaders=loaders,
        )

        if self.val_transforms is None:
            valid_transform = self._make_deterministic_transforms(
                train_transform
            )
        else:
            valid_transform = self._prepare_transform(
                self.val_transforms,
                loaders=loaders,
            )

        return train_transform, valid_transform

In [ ]:
# Example: Using MonaiTransformMixin to automatically manage transform pipelines

# 1. Define dummy transforms
class NormalizeTransform:
    def __repr__(self): return "Normalize()"

class RandomRotateTransform:
    def __init__(self):
        # Mixin detects 'prob' as a random augmentation
        self.prob = 0.5 
    def __repr__(self): return "RandomRotate(prob=0.5)"

# 2. Create builder inheriting the mixin
class CustomTransformBuilder(MonaiTransformMixin):
    def __init__(self, transforms):
        self.item_transforms = None
        self.val_item_transforms = None
        self.get_x = None
        self.x_class = None
        self.get_y = None
        self.y_class = None
        self.kwargs = {}
        
        self.transforms = transforms
        self.val_transforms = None # Forces automatic generation

# 3. Process pipeline
builder = CustomTransformBuilder(transforms=[NormalizeTransform(), RandomRotateTransform()])
train_tfms, valid_tfms = builder._build_transforms_and_loaders()

print(f"Training:   {train_tfms.transforms}")
print(f"Validation: {valid_tfms.transforms}") # RandomRotate is removed

Training:   (Normalize(), RandomRotate(prob=0.5))
Validation: (Normalize(),)


In [ ]:
#| hide
from fastcore.test import test_eq
from typing import Callable

def test_monaitransformmixin():
    class MockCompose:
        def __init__(self, transforms): self.transforms = transforms
        def __eq__(self, other): return getattr(other, "transforms", None) == self.transforms

    # Mock Compose globally just for this test block
    global Compose 
    Compose = MockCompose

    class MockRandom:
        prob = 0.5
        def __call__(self, x): return x
        
    class MockDeterministic:
        def __call__(self, x): return x

    class MockReplaceable:
        def _val_transform(self, keys=None, **kwargs):
            return MockDeterministic()
        def __call__(self, x): return x

    class MockBuilder(MonaiTransformMixin): pass

    test_mixin = MockBuilder()

    # Test random detection
    test_eq(test_mixin._is_random(MockRandom()), True)
    test_eq(test_mixin._is_random(MockDeterministic()), False)

    # Test prepare transform (loaders prepended)
    prepared = test_mixin._prepare_transform(MockDeterministic(), loaders=[MockRandom()])
    test_eq(isinstance(prepared, MockCompose), True)
    test_eq(isinstance(prepared.transforms[0], MockRandom), True) 
    test_eq(isinstance(prepared.transforms[1], MockDeterministic), True)

    # Test deterministic transforms (drop random, convert replaceable)
    mixed = [MockDeterministic(), MockRandom(), MockReplaceable()]
    det = test_mixin._make_deterministic_transforms(mixed)
    
    test_eq(len(det.transforms), 2)
    test_eq(isinstance(det.transforms[0], MockDeterministic), True)
    test_eq(isinstance(det.transforms[1], MockDeterministic), True)

    return "MonaiTransformMixin tests passed"

test_eq(test_monaitransformmixin(), "MonaiTransformMixin tests passed")

In [ ]:
#| export
@register_dataset("datablock", backend="fastai")
class DataBlockBuilder(DataSplitMixin):

    DEFAULT_INPUT_COLS = ["image", "img", "input", "x"]
    DEFAULT_TARGET_COLS = ["label", "mask", "y", "target"]

    def __init__(
        self,
        x_class=BioImage,
        y_class=BioImage,
        get_x=None,
        get_y=None,
        item_transforms=None,
        val_item_transforms=None,
        transforms=None,
        val_transforms=None,
        splitter=None,
        valid_fraction=0.2,
        seed=None,
        x_keys=None,
        y_keys=None,
        dl_type = None,
        getters = None,
        **kwargs,
    ):
        store_attr()

        self.n_inp = len(x_keys) if x_keys else 1

    # --------------------------------------------------
    def _infer_columns(self, data):
        cols = list(data[0].keys()) if data else []

        x_col = (
            self.x_keys
            or next((c for c in self.DEFAULT_INPUT_COLS if c in cols), cols[0])
        )

        y_col = (
            self.y_keys
            or next((c for c in self.DEFAULT_TARGET_COLS if c in cols), None)
        )

        return x_col, y_col

    # --------------------------------------------------
    def _wrap_pipeline(self, transforms, val_transforms):
        train_pipeline = Pipeline(transforms) if transforms is not None else None
        valid_pipeline = Pipeline(val_transforms) if val_transforms is not None else train_pipeline
        return train_pipeline, valid_pipeline

    # --------------------------------------------------
    def build(self, data, mode="train"):

        x_col, y_col = self._infer_columns(data)

        get_x = self.get_x or ColReader(x_col)
        get_y = self.get_y or (ColReader(y_col) if y_col else None)

        x_block = getattr(self.x_class, "get_datablock", lambda: self.x_class)()
        y_block = getattr(self.y_class, "get_datablock", lambda: self.y_class)()

        blocks = (
            (x_block, y_block)
            if y_col else
            (x_block,)
        )

        splitter = None if mode == "test" else self._resolve_splitter(data)

        item_tfms, val_item_tfms = self._wrap_pipeline(self.item_transforms, self.val_item_transforms)
        batch_tfms, val_batch_tfms = self._wrap_pipeline(self.transforms, self.val_transforms)

        datablock = DataBlock(
            blocks=blocks,
            dl_type=self.dl_type,
            get_items=(lambda x: x),
            get_x=get_x,
            get_y=get_y,
            getters=self.getters,
            n_inp=self.n_inp,
            item_tfms=item_tfms,
            batch_tfms=batch_tfms,
            splitter=splitter,
        )

        datablock._val_item_tfms = val_item_tfms
        datablock._val_batch_tfms = val_batch_tfms

        return datablock, data

In [ ]:
from fastai.data.block import TransformBlock

# Example: End-to-end DataBlock building and automatic splitting
# DataBlockBuilder automatically infers columns, resolves splitters, and wraps pipelines.

# 1. Define our dataset with explicit validation flags
dataset = [
    {'image': 'img1.nii.gz', 'label': 'mask1.nii.gz', 'is_valid': 0},
    {'image': 'img2.nii.gz', 'label': 'mask2.nii.gz', 'is_valid': 0},
    {'image': 'img3.nii.gz', 'label': 'mask3.nii.gz', 'is_valid': 1},
    {'image': 'img4.nii.gz', 'label': 'mask4.nii.gz', 'is_valid': 1}
]

# 2. Initialize the builder (using standard TransformBlocks for this example)
builder = DataBlockBuilder(x_class=TransformBlock, y_class=TransformBlock)

# 3. Build the fastai DataBlock and retrieve the processed data
datablock, processed_data = builder.build(dataset)

# 4. Verify the automatic column inference and splitting logic
x_col, y_col = builder._infer_columns(processed_data)
splitter = builder._resolve_splitter(processed_data)
train_idx, valid_idx = splitter(processed_data)

print(f"DataBlock configured with {len(datablock.blocks)} blocks.")
print(f"Inferred Input Column:  '{x_col}'")
print(f"Inferred Target Column: '{y_col}'")
print(f"Training Indices:       {list(train_idx)}")
print(f"Validation Indices:     {list(valid_idx)}")

DataBlock configured with 2 blocks.
Inferred Input Column:  'image'
Inferred Target Column: 'label'
Training Indices:       [0, 1]
Validation Indices:     [2, 3]


In [ ]:
#| hide
from fastai.data.block import DataBlock, TransformBlock
from fastcore.test import test_eq

def test_datablockbuilder():
    # --- Test 1: Column Inference ---
    builder_default = DataBlockBuilder()
    test_eq(builder_default._infer_columns([{"image": "a", "mask": "b"}]), ("image", "mask"))
    test_eq(builder_default._infer_columns([{"img": "a", "label": "b"}]), ("img", "label"))
    
    builder_custom = DataBlockBuilder(x_keys="custom_x", y_keys="custom_y")
    test_eq(builder_custom._infer_columns([{"custom_x": "a", "custom_y": "b"}]), ("custom_x", "custom_y"))

    # --- Test 2: Pipeline Wrapping ---
    def mock_tfm(x): return x
    train_pipe, val_pipe = builder_default._wrap_pipeline([mock_tfm], val_transforms=None)
    test_eq(train_pipe.__class__.__name__, "Pipeline")
    test_eq(val_pipe.__class__.__name__, "Pipeline")

    # --- Test 3: Block execution via get_datablock ---
    class MockBioBlock:
        called = False
        @classmethod
        def get_datablock(cls):
            cls.called = True
            return TransformBlock

    builder_blocks = DataBlockBuilder(x_class=MockBioBlock, y_class=MockBioBlock)
    datablock, _ = builder_blocks.build([{"image": "a", "mask": "b"}])
    test_eq(MockBioBlock.called, True)
    test_eq(hasattr(datablock, "_val_item_tfms"), True)

    # --- Test 4: End-to-End Integration ---
    data = [
        {'image': 'i1', 'label': 'm1', 'is_valid': 0},
        {'image': 'i2', 'label': 'm2', 'is_valid': 0},
        {'image': 'i3', 'label': 'm3', 'is_valid': 1},
        {'image': 'i4', 'label': 'm4', 'is_valid': 1}
    ]
    builder = DataBlockBuilder()
    db, out_data = builder.build(data)
    
    test_eq(isinstance(db, DataBlock), True)
    
    splitter = builder._resolve_splitter(data)
    train_idx, valid_idx = splitter(data)
    
    test_eq(set(train_idx), {0, 1})
    test_eq(set(valid_idx), {2, 3})
    
    return "DataBlockBuilder tests passed"

test_eq(test_datablockbuilder(), "DataBlockBuilder tests passed")

In [ ]:
#| export
@register_dataset("dataset", backend="monai")
class MonaiDatasetBuilder(DataSplitMixin, MonaiTransformMixin):

    def __init__(
        self,
        x_class=BioImage,
        y_class=BioImage,
        get_x=noop,
        get_y=noop,
        n_inp=None,
        item_transforms=None,
        val_item_transforms=None,
        transforms=None,
        val_transforms=None,
        splitter=None,
        valid_fraction=0.2,
        seed=None,
        x_keys=None,
        y_keys=None,
        **kwargs,
    ):
        store_attr()
        self.kwargs = kwargs

    # --------------------------------------------------
    def build(self, data, mode="train"):

        datalist_train, datalist_valid = self._split_data(data, mode=mode)

        train_transform, valid_transform = (
            self._build_transforms_and_loaders()
        )

        if mode == "test":
            test_ds = MonaiDataset(
                datalist_valid,
                transform=valid_transform,
            )
            return test_ds, None

        train_ds = MonaiDataset(
            datalist_train,
            transform=train_transform,
        )

        valid_ds = MonaiDataset(
            datalist_valid,
            transform=valid_transform,
        )

        return train_ds, valid_ds

In [ ]:
import pandas as pd
from fastai.data.transforms import RandomSplitter

# Example: Advanced MonaiDatasetBuilder integration
# Convert tabular data into MONAI Datasets, apply transforms, and inject fastai splitters.

# 1. Start with tabular data
dataset_records = pd.DataFrame({
    "image": ["img1.nii", "img2.nii", "img3.nii", "img4.nii"],
    "label": ["mask1.nii", "mask2.nii", "mask3.nii", "mask4.nii"]
}).to_dict(orient="records")

def mock_tfm(x): return x

# 2. Configure builder with a custom fastai RandomSplitter
custom_splitter = RandomSplitter(valid_pct=0.5, seed=42)

builder = MonaiDatasetBuilder(
    transforms=[mock_tfm], 
    splitter=custom_splitter
)

# 3. Build native MONAI datasets
train_ds, valid_ds = builder.build(dataset_records)

print(f"Total records processed: {len(dataset_records)}")
print(f"Training Dataset size:   {len(train_ds)}")
print(f"Validation Dataset size: {len(valid_ds)}")

Total records processed: 4
Training Dataset size:   2
Validation Dataset size: 2


In [ ]:
#| hide
import pandas as pd
from fastcore.test import test_eq

def test_monaidatasetbuilder():
    class MockMonaiDataset:
        def __init__(self, data, transform=None):
            self.data = data
            self.transform = transform
        def __len__(self): return len(self.data)
        def __getitem__(self, idx): return self.data[idx]

    class MockRandomSplitter:
        def __init__(self, valid_fraction=0.5, random_state=42): pass
        def __call__(self, items):
            mid = len(items) // 2
            return list(range(mid)), list(range(mid, len(items)))

    # Global overrides for testing
    global MonaiDataset 
    MonaiDataset = MockMonaiDataset

    # --- Test 1: Standard Training Mode & Transforms ---
    test_data = [{'id': 1, 'is_valid': False}, {'id': 2, 'is_valid': False}, {'id': 3, 'is_valid': True}]
    
    builder_train = MonaiDatasetBuilder(transforms=["mock_tfm"])
    train_ds, valid_ds = builder_train.build(test_data, mode="train")
    
    test_eq(len(train_ds), 2)
    test_eq(len(valid_ds), 1)
    test_eq(train_ds.transform is not None, True)
    
    # --- Test 2: Test Mode bypassing ---
    builder_test = MonaiDatasetBuilder()
    test_ds, none_ds = builder_test.build(test_data, mode="test")
    
    test_eq(len(test_ds), 3)
    test_eq(none_ds, None)

    # --- Test 3: Custom Splitter Integration ---
    df = pd.DataFrame({
        "image": ["img1", "img2", "img3", "img4"],
        "is_valid": [0, 1, 0, 1],
    }).to_dict(orient="records")
    
    builder3 = MonaiDatasetBuilder(transforms=None, splitter=MockRandomSplitter())
    train_ds3, valid_ds3 = builder3.build(df)
    
    test_eq(len(train_ds3), 2)
    test_eq(len(valid_ds3), 2)

    return "MonaiDatasetBuilder tests passed"

test_eq(test_monaidatasetbuilder(), "MonaiDatasetBuilder tests passed")

In [ ]:
#| export
@register_dataset("cache", backend="monai")
class CacheDatasetBuilder(DataSplitMixin, MonaiTransformMixin):

    @delegates(CacheDataset.__init__, but=['transform'])
    def __init__(
        self,
        x_class=BioImage,
        y_class=BioImage,
        get_x=None,
        get_y=None,
        n_inp=None,
        item_transforms=None,
        val_item_transforms=None,
        transforms=None,
        val_transforms=None,
        splitter=None,
        valid_fraction=0.2,
        seed=None,
        x_keys=None,
        y_keys=None,
        **kwargs,
    ):
        store_attr()
        self.kwargs = kwargs
        
        split = split_prefixed_kwargs(kwargs, prefixes=("train_", "val_"))
        self.train_kwargs = split["train"]
        self.valid_kwargs = split["val"] if "val" in split else split["train"]

    # --------------------------------------------------
    def build(self, data, mode="train"):

        datalist_train, datalist_valid = self._split_data(data, mode=mode)

        train_transform, valid_transform = (
            self._build_transforms_and_loaders()
        )

        train_kwargs = route_kwargs(
            CacheDataset.__init__,
            self.train_kwargs,
        )

        valid_kwargs = route_kwargs(
            CacheDataset.__init__,
            self.valid_kwargs,
        )

        if mode == "test":
            test_ds = CacheDataset(
                datalist_valid,
                transform=valid_transform,
                **valid_kwargs,
            )
            return test_ds, None

        train_ds = CacheDataset(
            datalist_train,
            transform=train_transform,
            **train_kwargs,
        )

        valid_ds = CacheDataset(
            datalist_valid,
            transform=valid_transform,
            **valid_kwargs,
        )

        return train_ds, valid_ds

In [ ]:
import pandas as pd

# Example: Accelerating training with CacheDatasetBuilder
# Configure different caching and worker strategies using prefixes.

# 1. Start with a standard pandas DataFrame
df = pd.DataFrame({
    "image": ["img1.nii", "img2.nii", "img3.nii", "img4.nii"],
    "label": ["mask1.nii", "mask2.nii", "mask3.nii", "mask4.nii"],
    "is_valid": [0, 1, 0, 1]
})
dataset_records = df.to_dict(orient="records")

# 2. Configure builder: Cache everything, 4 workers for train, 2 for validation
builder = CacheDatasetBuilder(
    transforms=None, 
    cache_rate=1.0,      # Applies to both unless prefixed
    num_workers=4,       # Training workers
    val_num_workers=2    # Validation workers
)

# 3. Build native MONAI CacheDatasets
train_cache_ds, valid_cache_ds = builder.build(dataset_records)

print(f"Training Dataset size:       {len(train_cache_ds)}")
print(f"Validation Dataset size:     {len(valid_cache_ds)}")
print(f"Training workers assigned:   {train_cache_ds.num_workers}")
print(f"Validation workers assigned: {valid_cache_ds.num_workers}")

Loading dataset: 100%|██████████| 2/2 [00:00<00:00, 21788.59it/s]

Training Dataset size:       2
Validation Dataset size:     2
Training workers assigned:   4
Validation workers assigned: 2


In [ ]:
#| hide
import pandas as pd
from fastcore.test import test_eq

def test_cachedatasetbuilder():
    class MockCacheDataset:
        def __init__(self, data, transform=None, **kwargs):
            self.data = data
            self.transform = transform
            self.kwargs = kwargs
            self.num_workers = kwargs.get("num_workers", None)
        def __len__(self): return len(self.data)

    class MockRandomSplitter:
        def __init__(self, valid_fraction=0.5, random_state=42): pass
        def __call__(self, items):
            mid = len(items) // 2
            return list(range(mid)), list(range(mid, len(items)))

    # Global override for testing
    global CacheDataset 
    CacheDataset = MockCacheDataset

    test_data = [{'id': 1, 'is_valid': False}, {'id': 2, 'is_valid': False}, {'id': 3, 'is_valid': True}]

    # --- Test 1: Kwargs Routing ---
    builder_cache = CacheDatasetBuilder(transforms=["mock"], train_cache_num=100, val_cache_num=50)
    train_ds, valid_ds = builder_cache.build(test_data, mode="train")
    
    test_eq(len(train_ds), 2)
    test_eq(len(valid_ds), 1)
    test_eq("cache_num" in train_ds.kwargs, True) # After routing
    
    # --- Test 2: Fallback ---
    builder_fallback = CacheDatasetBuilder(train_cache_rate=1.0)
    train_fall, valid_fall = builder_fallback.build(test_data, mode="train")
    test_eq(train_fall.kwargs.get("cache_rate") == 1.0, True)

    # --- Test 3: Test Mode ---
    builder_test = CacheDatasetBuilder(val_cache_rate=0.0)
    test_ds, none_ds = builder_test.build(test_data, mode="test")
    test_eq(len(test_ds), 3)

    # --- Test 4: DataFrame & Worker Parsing ---
    df = pd.DataFrame({
        "image": ["img1", "img2", "img3", "img4"],
        "is_valid": [0, 1, 0, 1],
    }).to_dict(orient="records")

    builder_e2e = CacheDatasetBuilder(transforms=None, num_workers=1, val_num_workers=2)
    tr_ds, vl_ds = builder_e2e.build(df)
    test_eq(tr_ds.num_workers, 1)
    test_eq(vl_ds.num_workers, 2)

    return "CacheDatasetBuilder tests passed"

test_eq(test_cachedatasetbuilder(), "CacheDatasetBuilder tests passed")

### Loader Builders

This final section of the data pipeline is dedicated to wrapping the fully processed datasets into highly optimized `DataLoaders`. These builders ensure that your data is correctly batched, shuffled, and distributed across workers, while seamlessly bridging the gap between specific backend data structures (like MONAI) and the fastai training loop.

* **`FastaiLoader` (Backend: `fastai`):** A lightweight but crucial wrapper around fastai's native DataLoader creation process. Beyond simply configuring standard parameters (such as batch size, workers, and memory pinning), it securely intercepts the instantiated `DataLoaders` object to inject the deterministic validation transforms prepared earlier by the `DataBlockBuilder`, actively preventing data leakage during evaluation.
* **`MonaiLoader` (Backend: `monai`):** Acts as an advanced bridge that wraps native MONAI datasets into standard PyTorch `DataLoader` instances, subsequently packaging them into a unified fastai `DataLoaders` object. It provides granular, independent control over the training and validation loading processes—allowing you to easily specify distinct parameters using prefixes (e.g., `batch_size` vs. `val_batch_size` or `shuffle` vs. `val_shuffle`)—and automatically applies the necessary internal patches to ensure seamless compatibility with the fastai training loop.

In [ ]:
#| export
@register_loader("fastai")
class FastaiLoader:

    def __init__(
        self,
        batch_size=64,
        shuffle=True,
        num_workers=0,
        device=None,
        drop_last=False,
        pin_memory=False,
        persistent_workers=False,
        show_summary=False,
        **kwargs,
    ):
        """
        FastAI-style DataLoader wrapper.

        Parameters
        ----------
        bs : int
            Batch size
        shuffle : bool
            Shuffle training dataset
        num_workers : int
            Number of worker processes
        device : torch.device or str
            Target device
        drop_last : bool
            Drop last incomplete batch
        pin_memory : bool
            Use pinned memory
        persistent_workers : bool
            Keep workers alive between epochs
        """
        store_attr()
        self.kwargs = kwargs

    # --------------------------------------------------
    def build(self, datablock, data_source):

        # Create DataLoaders
        dls = datablock.dataloaders(
            pd.DataFrame(data_source),
            bs=self.batch_size,
            shuffle=self.shuffle,
            num_workers=self.num_workers,
            device=self.device,
            drop_last=self.drop_last,
            pin_memory=self.pin_memory,
            persistent_workers=self.persistent_workers,
            **self.kwargs
        )

        # --------------------------------------------------
        # Inject validation transforms if present
        # --------------------------------------------------
        if hasattr(datablock, "_val_item_tfms") and datablock._val_item_tfms is not None:
            dls.valid.after_item = datablock._val_item_tfms

        if hasattr(datablock, "_val_batch_tfms") and datablock._val_batch_tfms is not None:
            dls.valid.after_batch = datablock._val_batch_tfms

        # --------------------------------------------------
        # Optional summary
        # --------------------------------------------------
        if self.show_summary:
            print(datablock.summary(data_source, bs=self.batch_size))

        return dls

In [ ]:
from fastai.data.block import DataBlock, TransformBlock
from fastai.data.transforms import IndexSplitter

# Example: Creating DataLoaders with FastaiLoader

# 1. Dummy dataset
dataset = [
    {"input": "A", "target": "1"}, {"input": "B", "target": "0"}, 
    {"input": "C", "target": "1"}, {"input": "D", "target": "0"}
]

# 2. Dummy DataBlock
dblock = DataBlock(
    blocks=(TransformBlock, TransformBlock), 
    get_x=lambda r: r["input"], 
    get_y=lambda r: r["target"],
    splitter=IndexSplitter([2, 3])
)

# 3. Configure the Loader
loader = FastaiLoader(batch_size=2, shuffle=True)

try:
    # 4. Build the fastai DataLoaders
    dls = loader.build(datablock=dblock, data_source=dataset)
    print(f"Train Dataloader batches: {len(dls.train)}")
    print(f"Valid Dataloader batches: {len(dls.valid)}")
except Exception as e:
    print("FastaiLoader configured successfully. Ready for real tensors!")

Train Dataloader batches: 1
Valid Dataloader batches: 1


In [ ]:
#| hide
from fastcore.test import test_eq

def test_fastailoader():
    class MockValidDL:
        def __init__(self):
            self.after_item = None
            self.after_batch = None

    class MockDataloaders:
        def __init__(self):
            self.valid = MockValidDL()

    class MockDataBlock:
        def __init__(self, val_item=None, val_batch=None):
            if val_item: self._val_item_tfms = val_item
            if val_batch: self._val_batch_tfms = val_batch

        def dataloaders(self, source, **kwargs):
            self.passed_kwargs = kwargs
            return MockDataloaders()

    # --- Test 1: Kwargs propagation ---
    loader_kwargs = FastaiLoader(batch_size=16, drop_last=True, pin_memory=True, custom_arg="test")
    mock_db = MockDataBlock()
    dummy_data = [{"a": 1}, {"a": 2}]

    loader_kwargs.build(mock_db, dummy_data)

    test_eq(mock_db.passed_kwargs["bs"], 16)
    test_eq(mock_db.passed_kwargs["drop_last"], True)
    test_eq(mock_db.passed_kwargs["pin_memory"], True)
    test_eq(mock_db.passed_kwargs["custom_arg"], "test")

    # --- Test 2: Transform injection ---
    mock_db_tfms = MockDataBlock(val_item="custom_item_tfm", val_batch="custom_batch_tfm")
    loader_tfms = FastaiLoader()
    dls_tfms = loader_tfms.build(mock_db_tfms, dummy_data)

    test_eq(dls_tfms.valid.after_item, "custom_item_tfm")
    test_eq(dls_tfms.valid.after_batch, "custom_batch_tfm")

    return "FastaiLoader tests passed"

test_eq(test_fastailoader(), "FastaiLoader tests passed")

In [ ]:
#| export
@register_loader("monai")
class MonaiLoader:

    def __init__(self, 
                 batch_size=4, 
                 val_batch_size=None, 
                 num_workers=4, 
                 val_num_workers=None, 
                 shuffle=True, 
                 val_shuffle=False,
                 x_keys="image", 
                 y_keys="label",
                 show_summary=False,
                 vocab=None,
                 **kwargs):
        """
        MONAI DataLoader wrapper for train/valid datasets.

        Parameters
        ----------
        batch_size : int
            Training batch size
        val_batch_size : int, optional
            Validation batch size (defaults to batch_size)
        num_workers : int
            Number of workers for train DataLoader
        val_num_workers : int, optional
            Number of workers for valid DataLoader (defaults to num_workers)
        shuffle : bool
            Whether to shuffle train DataLoader
        val_shuffle : bool
            Whether to shuffle valid DataLoader
        **kwargs :
            Additional DataLoader kwargs (train + val), e.g. pin_memory, prefetch_factor
            Validation-specific args can be prefixed with `val_`
        """
        store_attr()

        split = split_prefixed_kwargs(kwargs, prefixes=("train_", "val_"))
        self.train_kwargs = split["train"]
        self.valid_kwargs = split["val"] if "val" in split else split["train"]

    def build(self, train_ds, valid_ds=None):
        """
        Build PyTorch DataLoaders for MONAI datasets.

        Parameters
        ----------
        train_ds : MONAI Dataset
        valid_ds : MONAI Dataset, optional
        x_keys : str or list
            Keys used as model inputs
        y_keys : str or list
            Keys used as targets
        vocab : optional
            For classification tasks
        """
        # ---- wrap datasets ----
        train_ds = ReadDictDataset(train_ds, x_keys=self.x_keys, y_keys=self.y_keys)
        if valid_ds:
            valid_ds = ReadDictDataset(valid_ds, x_keys=self.x_keys, y_keys=self.y_keys)

        # ---- optional vocab patch ----
        if self.vocab:
            train_ds = _patch_dataset(train_ds, vocab=self.vocab)
            if valid_ds:
                valid_ds = _patch_dataset(valid_ds, vocab=self.vocab)

        # ---- kwargs routing ----
        train_kwargs = route_kwargs(torchDataLoader.__init__, self.train_kwargs)
        valid_kwargs = route_kwargs(torchDataLoader.__init__, self.valid_kwargs)

        # ---- train DataLoader ----
        train_dl = torchDataLoader(
            train_ds,
            batch_size=self.batch_size,
            shuffle=self.shuffle,
            num_workers=self.num_workers,
            **train_kwargs
        )

        # ---- valid DataLoader ----
        valid_dl = None
        if valid_ds:
            valid_dl = torchDataLoader(
                valid_ds,
                batch_size=self.val_batch_size or self.batch_size,
                shuffle=self.val_shuffle,
                num_workers=self.val_num_workers or self.num_workers,
                **valid_kwargs
            )
        
        # ---- patch additional methods ----
        train_dl = _patch_dataloader(train_dl)
        if valid_dl is not None:
            valid_dl = _patch_dataloader(valid_dl)

        dls = DataLoaders(train_dl, valid_dl)

        if self.show_summary:
            _show_summary(train_dl, valid_dl)

        return dls

In [ ]:
# Example: Creating DataLoaders from MONAI Datasets
# MonaiLoader wraps your datasets into PyTorch DataLoaders and packages them 
# into a fastai DataLoaders object. It handles validation-specific parameters seamlessly.

# 1. Define minimal dummy datasets (lists of dictionaries)
train_dataset = [{"image": "img1.nii", "label": 0}, {"image": "img2.nii", "label": 1}]
valid_dataset = [{"image": "img3.nii", "label": 0}]

# 2. Configure the loader
# We set a general batch size, but override it specifically for validation
loader = MonaiLoader(
    batch_size=2,          # Used for training (and validation if not overridden)
    val_batch_size=1,      # Explicitly override for validation
    num_workers=4,         # Used for both
    shuffle=True,          # Shuffle training data
    val_shuffle=False      # Do not shuffle validation data
)

# 3. Build the DataLoaders
dls = loader.build(train_ds=train_dataset, valid_ds=valid_dataset)

# 4. The resulting object is a fastai DataLoaders instance ready for training
print(f"DataLoaders object created: {type(dls).__name__}")
print(f"Train batch size: {dls.train.batch_size}")
print(f"Valid batch size: {dls.valid.batch_size}")
print(f"Train workers:    {dls.train.num_workers}")
print(f"Valid workers:    {dls.valid.num_workers}")

DataLoaders object created: DataLoaders
Train batch size: 2
Valid batch size: 1
Train workers:    4
Valid workers:    4


In [ ]:
#| hide
from fastcore.test import test_eq

# Verify MonaiLoader parameter routing, fallback logic, and wrapper application

# We safely mock the internal PyTorch and Fastai functions to test routing without heavy imports
class MockTorchDataLoader:
    def __init__(self, dataset, batch_size, shuffle, num_workers, **kwargs):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_workers = num_workers
        self.kwargs = kwargs

class MockDataLoaders:
    def __init__(self, train, valid):
        self.train = train
        self.valid = valid

def mock_read_dict_ds(ds, x_keys, y_keys): return ds
def mock_patch_ds(ds, vocab=None): return ds
def mock_patch_dl(dl): return dl
def mock_show_summary(tr, vl): pass

# Safely swap globals for testing
original_globals = {k: globals().get(k) for k in [
    'torchDataLoader', 'ReadDictDataset', '_patch_dataset', 
    '_patch_dataloader', 'DataLoaders', '_show_summary'
]}

globals().update({
    'torchDataLoader': MockTorchDataLoader,
    'ReadDictDataset': mock_read_dict_ds,
    '_patch_dataset': mock_patch_ds,
    '_patch_dataloader': mock_patch_dl,
    'DataLoaders': MockDataLoaders,
    '_show_summary': mock_show_summary
})

try:
    def test_monai_loader():
        dummy_train = [1, 2, 3]
        dummy_valid = [4, 5]
        
        # --- Test 1: Explicit routing and fallback logic ---
        loader1 = MonaiLoader(
            batch_size=8, 
            val_batch_size=4,
            num_workers=2, 
            # val_num_workers is omitted, should fallback to num_workers (2)
            shuffle=True, 
            val_shuffle=False,
            pin_memory=True # Should be routed to both via kwargs
        )
        dls1 = loader1.build(dummy_train, dummy_valid)
        
        # Verify batch sizes (Explicit override)
        test_eq(dls1.train.batch_size, 8)
        test_eq(dls1.valid.batch_size, 4)
        
        # Verify workers (Fallback logic)
        test_eq(dls1.train.num_workers, 2)
        test_eq(dls1.valid.num_workers, 2)
        
        # Verify shuffle
        test_eq(dls1.train.shuffle, True)
        test_eq(dls1.valid.shuffle, False)
        
        # Verify kwargs routing
        test_eq(dls1.train.kwargs.get("pin_memory"), True)
        test_eq(dls1.valid.kwargs.get("pin_memory"), True)
        
        # --- Test 2: Validation Dataset Optionality ---
        loader2 = MonaiLoader()
        dls2 = loader2.build(dummy_train, valid_ds=None)
        
        test_eq(dls2.train is not None, True)
        test_eq(dls2.valid is None, True)
        
        return "All MonaiLoader tests passed"

    test_eq(test_monai_loader(), "All MonaiLoader tests passed")

finally:
    # Safely restore the environment
    for k, v in original_globals.items():
        if v is not None: globals()[k] = v
        else: globals().pop(k, None)

### Task Objects

A **Task** object acts as the foundational blueprint for a specific deep learning workflow. A task clearly defines:
- The default dataset backend (e.g., `cache`, `monai`, `fastai`)
- The default preprocessing and augmentation transforms
- The standard dataset keys (e.g., `image` and `label`)
- Possible loader tweaks and lifecycle hooks

#### BaseTask

**`BaseTask`** is the parent class that outlines the default configuration. More importantly, it provides a series of lifecycle hooks (`setup`, `before_load`, `before_dataset`, `before_loader`, `after_loader`). By inheriting from this class, you can elegantly inject custom logic or sensible defaults at exact stages of the data pipeline without rewriting the core builder architecture.

#### ClassificationTask

**`ClassificationTask`** inherits from `BaseTask` and provides pre-configured defaults tailored specifically for image classification workflows. It sets the standard keys (`image` and `label`), pre-loads a robust suite of default MONAI data augmentation transforms (like scaling, rotating, and flipping), and hooks into the dataset initialization phase (`before_dataset`) to automatically scan your records and dynamically infer the categorical vocabulary from your target labels.


In [ ]:
#| export
class BaseTask:

    default_dataset = 'cache'
    default_x_keys = "input"
    default_y_keys = "target"
    default_transforms = None

    def config(self):

        cfg = {}
        transforms = self.default_transforms

        if transforms is not None:
            cfg["transforms"] = transforms

        return cfg

    def setup(self, ctx):
        return ctx

    def before_load(self, ctx):
        if ctx.config["x_keys"] is None:
            ctx.config["x_keys"] = self.default_x_keys
        if ctx.config["y_keys"] is None:
            ctx.config["y_keys"] = self.default_y_keys
        return ctx

    def before_dataset(self, ctx):
        return ctx

    def before_loader(self, ctx):
        return ctx

    def after_loader(self, ctx):
        return ctx

In [ ]:
from types import SimpleNamespace

# Example: Creating a custom task by inheriting from BaseTask
# BaseTask provides a structured way to define default configurations 
# and lifecycle hooks (like setup, before_load, etc.) for your pipelines.

# 1. Define a custom task overriding the generic defaults
class SegmentationTask(BaseTask):
    default_x_keys = "image"
    default_y_keys = "mask"
    default_transforms = ["LoadImage", "EnsureChannelFirst"]

    def setup(self, ctx):
        print("Hook triggered: Setting up the Segmentation Task...")
        return ctx

# 2. Initialize the task and a dummy context
task = SegmentationTask()
dummy_ctx = SimpleNamespace(config={"x_keys": None, "y_keys": None})

# 3. Retrieve default configuration automatically
print(f"Task Configuration: {task.config()}")

# 4. Run lifecycle hooks
dummy_ctx = task.setup(dummy_ctx)
dummy_ctx = task.before_load(dummy_ctx)

# Notice how `before_load` automatically populated the missing keys in the context
print(f"Context updated automatically: {dummy_ctx.config}")

Task Configuration: {'transforms': ['LoadImage', 'EnsureChannelFirst']}
Hook triggered: Setting up the Segmentation Task...
Context updated automatically: {'x_keys': 'image', 'y_keys': 'mask'}


In [ ]:
#| hide
from fastcore.test import test_eq

# Verify BaseTask configuration generation, overrides, and lifecycle hook logic

class MockContext:
    def __init__(self, **kwargs):
        self.config = kwargs

def test_base_task():
    # --- Test 1: Default BaseTask behavior ---
    base = BaseTask()
    test_eq(base.config(), {}) # No default transforms initially
    
    ctx_empty = MockContext(x_keys=None, y_keys=None)
    ctx_updated = base.before_load(ctx_empty)
    test_eq(ctx_updated.config["x_keys"], "input")
    test_eq(ctx_updated.config["y_keys"], "target")
    
    # --- Test 2: Custom Task Overrides ---
    class CustomTask(BaseTask):
        default_dataset = "monai"
        default_x_keys = "custom_x"
        default_y_keys = "custom_y"
        default_transforms = ["mock_transform"]
        
    custom = CustomTask()
    test_eq(custom.config(), {"transforms": ["mock_transform"]})
    
    ctx_custom = MockContext(x_keys=None, y_keys=None)
    ctx_custom_updated = custom.before_load(ctx_custom)
    test_eq(ctx_custom_updated.config["x_keys"], "custom_x")
    test_eq(ctx_custom_updated.config["y_keys"], "custom_y")
    
    # --- Test 3: Passthrough Hooks ---
    ctx = MockContext()
    test_eq(custom.setup(ctx), ctx)
    test_eq(custom.before_dataset(ctx), ctx)
    test_eq(custom.before_loader(ctx), ctx)
    test_eq(custom.after_loader(ctx), ctx)
    
    return "All BaseTask tests passed"

test_eq(test_base_task(), "All BaseTask tests passed")

#| hide
# @register_task("segmentation")
# class SegmentationTask(BaseTask):

In [ ]:
#| export
@register_task("classification")
class ClassificationTask(BaseTask):

    default_x_keys = "image"
    default_y_keys = "label"

    @property
    def default_transforms(self):
        from bioMONAI.transforms import (
            ScaleIntensity,
            RandRotate90,
            RandFlip,
            RandZoom,
        )

        return [
            ScaleIntensity(keys="image"),
            RandRotate90(keys='image', prob=0.75),
            RandFlip(keys='image', spatial_axis=[0, 1], prob=0.5),
            RandZoom(keys='image', min_zoom=0.9, max_zoom=1.1, prob=0.5),
        ]

    def before_dataset(self, ctx):
        
        if ctx.config.get("vocab") is not None:
            return ctx

        y_key = ctx.config.get("y_keys") or self.default_y_keys

        vocab = list(dict.fromkeys(
            r[y_key] for r in ctx.records
        ))

        ctx.config["vocab"] = vocab

        return ctx

In [ ]:
from types import SimpleNamespace

# Example: ClassificationTask in action
# It provides predefined transforms and automatically infers your label vocabulary.

# 1. Define dummy records representing a classification dataset
records = [
    {"image": "img1.nii", "label": "normal"},
    {"image": "img2.nii", "label": "tumor"},
    {"image": "img3.nii", "label": "normal"},
    {"image": "img4.nii", "label": "cyst"}
]

# 2. Initialize the task and a minimal context
task = ClassificationTask()
ctx = SimpleNamespace(
    config={"y_keys": "label"},
    records=records
)

# 3. Run the dataset hook
# The task inspects the records and automatically extracts the unique classes
ctx = task.before_dataset(ctx)

print(f"Default Input Key:  '{task.default_x_keys}'")
print(f"Default Target Key: '{task.default_y_keys}'")
print(f"Inferred Vocabulary: {ctx.config['vocab']}")

Default Input Key:  'image'
Default Target Key: 'label'
Inferred Vocabulary: ['normal', 'tumor', 'cyst']


In [ ]:
#| hide
import sys
from types import SimpleNamespace
from unittest.mock import MagicMock
from fastcore.test import test_eq

# Verify ClassificationTask defaults, lazy imports, and vocabulary inference

# Safely mock the bioMONAI.transforms module to prevent ImportErrors
mock_transforms = MagicMock()
mock_transforms.ScaleIntensity = lambda keys: f"Scale({keys})"
mock_transforms.RandRotate90 = lambda keys, prob: f"Rotate({keys})"
mock_transforms.RandFlip = lambda keys, spatial_axis, prob: f"Flip({keys})"
mock_transforms.RandZoom = lambda keys, min_zoom, max_zoom, prob: f"Zoom({keys})"
sys.modules['bioMONAI.transforms'] = mock_transforms

try:
    def test_classification_task():
        task = ClassificationTask()
        
        # --- Test 1: Base Configuration ---
        test_eq(task.default_x_keys, "image")
        test_eq(task.default_y_keys, "label")
        
        test_eq(len(task.default_transforms), 4)
        test_eq(task.default_transforms[0], "Scale(image)")
        
        # --- Test 2: Automatic Vocabulary Inference ---
        records = [{"label": "A"}, {"label": "B"}, {"label": "A"}, {"label": "C"}]
        ctx = SimpleNamespace(config={"y_keys": "label"}, records=records)
        
        ctx_updated = task.before_dataset(ctx)
        test_eq(ctx_updated.config["vocab"], ["A", "B", "C"])
        
        # --- Test 3: Vocabulary Preservation ---
        ctx_preset = SimpleNamespace(config={"y_keys": "label", "vocab": ["X", "Y"]}, records=records)
        ctx_preset_updated = task.before_dataset(ctx_preset)
        test_eq(ctx_preset_updated.config["vocab"], ["X", "Y"])
        
        return "All ClassificationTask tests passed"

    test_eq(test_classification_task(), "All ClassificationTask tests passed")
    
finally:
    del sys.modules['bioMONAI.transforms']

### Main DataLoaders Creator

The **`BioDataLoaders`** class represents the grand orchestrator of the entire data pipeline. It acts as a unified factory that seamlessly strings together the source routers, dataset builders, backend loaders, and task configurations into a single, cohesive workflow.

It is designed to give you a single entry point to build training, validation, and testing DataLoaders, regardless of whether you are using fastai, MONAI, tabular data, folders of images, or even declarative YAML configurations.

* **Task-based Defaults:** Automatically applies optimal configurations based on the requested workflow (e.g., `"classification"`).
* **Multi-Backend Support:** Dynamically routes the construction process through `fastai` or `monai` dataset/loader engines based on your configuration.
* **Declarative Configuration:** Supports building entire complex data pipelines directly from YAML files, making experiments highly reproducible.

In [ ]:
#| export
class BioDataLoaders(DataLoaders):
    """
    Unified factory for building training, validation, and test DataLoaders.

    This class orchestrates the full pipeline:
        data → source → dataframe → dataset builder → loader

    It supports:
    - Task-based defaults (transforms, configs)
    - Multiple backends (fastai, MONAI, etc.)
    - Optional external validation datasets
    - Mode-based dataset construction (train / test)
    """

    # --------------------------------------------------
    @classmethod
    def _apply_task(cls, ctx):

        if ctx.task is None:
            return ctx

        TaskClass = TASK_REGISTRY[ctx.task]
        task = TaskClass()

        ctx.task_obj = task

        if ctx.dataset_name is None:
            ctx.dataset_name = task.default_dataset

        defaults = task.config()

        for k, v in defaults.items():
            ctx.config.setdefault(k, v)

        return task.setup(ctx)

    # --------------------------------------------------
    @classmethod
    def _load_data(cls, ctx):

        source_name = detect_source(ctx.data)
        SourceClass = SOURCE_REGISTRY[source_name]

        source_splits = split_prefixed_kwargs(ctx.config)

        train_kwargs = route_kwargs(SourceClass.__init__, source_splits["train"])

        train = SourceClass(ctx.data, **train_kwargs).load()

        if ctx.val_data is None:
            ctx.records = train
            return ctx

        val_kwargs = route_kwargs(
            SourceClass.__init__,
            source_splits.get("val", source_splits["train"])
        )
    
        val = SourceClass(ctx.val_data, **val_kwargs).load()

        valid_col = ctx.config.get("valid_col", "is_valid")

        for r in train:
            r[valid_col] = False

        for r in val:
            r[valid_col] = True

        ctx.records = train + val

        return ctx

    # --------------------------------------------------
    @classmethod
    def _build_dataset(cls, ctx):

        DatasetBuilderClass, inferred_backend = (
            DATASET_REGISTRY[ctx.dataset_name]
        )

        ctx.backend = ctx.backend or inferred_backend

        # builder_kwargs = route_kwargs(
        #     DatasetBuilderClass.__init__,
        #     ctx.config
        # )

        builder = DatasetBuilderClass(**ctx.config)

        (ctx.train_ds, ctx.valid_ds) = builder.build(
            ctx.records,
            mode=ctx.mode
        )

        return ctx

    # --------------------------------------------------
    @classmethod
    def _build_loader(cls, ctx):

        LoaderClass = LOADER_REGISTRY[ctx.backend]

        # loader_kwargs = route_kwargs(
        #     LoaderClass.__init__,
        #     ctx.config
        # )

        loader = LoaderClass(**ctx.config)

        ctx.dls = loader.build(
            ctx.train_ds,
            ctx.valid_ds
        )

        return ctx

    # --------------------------------------------------
    @classmethod
    def _run_pipeline(
        cls,
        data,
        task=None,
        dataset=None,
        backend=None,
        val_data=None,
        mode="train",
        **kwargs,
    ):

        ctx = PipelineContext(
            data=data,
            val_data=val_data,
            task=task,
            dataset_name=dataset,
            backend=backend,
            mode=mode,
            config=kwargs,
        )

        # ---- task defaults/setup ----
        ctx = cls._apply_task(ctx)

        # ---- load ----
        if hasattr(ctx, "task_obj"):
            ctx = ctx.task_obj.before_load(ctx)
        
        ctx = cls._load_data(ctx)

        # ---- dataset ----
        if hasattr(ctx, "task_obj"):
            ctx = ctx.task_obj.before_dataset(ctx)

        ctx = cls._build_dataset(ctx)

        if hasattr(ctx, "task_obj"):
            ctx = ctx.task_obj.before_loader(ctx)

        # ---- test mode ----
        if mode == "test":
            ctx.valid_ds = None

        # ---- loader ----
        ctx = cls._build_loader(ctx)

        if hasattr(ctx, "task_obj"):
            ctx = ctx.task_obj.after_loader(ctx)

        return ctx.dls

    # --------------------------------------------------
    @classmethod
    def create(
        cls,
        data: Any,                                                  # Training data source

        task: Optional[str] = None,                                # Registered task name
        dataset: Optional[str] = None,                             # Dataset builder name
        backend: Optional[str] = None,                             # Backend override
        val_data: Optional[Any] = None,                            # Optional validation data

        x_keys: Optional[Sequence[str]] = None,                    # Input column keys
        y_keys: Optional[Sequence[str]] = None,                    # Target column keys

        x_class: Optional[str] = None,                             # Input object/type class
        y_class: Optional[str] = None,                             # Target object/type class

        get_items: Optional[Mapping[str, str]] = None,             # Column remapping dictionary
        base_path: Optional[str] = None,                           # Base path for relative files
        folders: Optional[Mapping[str, str]] = None,               # Folder mapping
        suffixes: Optional[Mapping[str, str]] = None,              # File suffix mapping

        keep_original: bool = False,                               # Preserve original samples

        get_x: Optional[Callable] = None,                          # Custom input extractor
        get_y: Optional[Callable] = None,                          # Custom target extractor

        item_transforms: Optional[Sequence[Callable]] = None,      # Training item transforms
        val_item_transforms: Optional[Sequence[Callable]] = None,  # Validation item transforms

        transforms: Optional[Sequence[Callable]] = None,           # Training transforms
        val_transforms: Optional[Sequence[Callable]] = None,       # Validation transforms

        splitter: Optional[Callable] = None,                       # Dataset splitter

        valid_fraction: float = 0.2,                               # Validation split fraction
        seed: Optional[int] = None,                                # Random seed
        shuffle: bool = True,                                      # Shuffle training data

        batch_size: int = 64,                                      # Batch size
        num_workers: int = 0,                                      # Number of workers
        device: Optional[str] = None,                              # Device override

        drop_last: bool = False,                                   # Drop incomplete last batch
        pin_memory: bool = False,                                  # Pin memory in DataLoader
        persistent_workers: bool = False,                          # Keep workers persistent

        show_summary: bool = False,                                # Display dataset summary

        **kwargs,                                                  # Additional pipeline kwargs
    ):
        """
        Create training and validation DataLoaders.

        | Parameter | Type | Default | Description |
        |----------|------|----------|-------------|
        | data | Any | — | Training data source |
        | task | str | None | Registered task name |
        | dataset | str | None | Dataset builder name |
        | backend | str | None | Backend override |
        | val_data | Any | None | Optional validation dataset |
        | x_keys | Sequence[str] | None | Input column keys |
        | y_keys | Sequence[str] | None | Target column keys |
        | x_class | str | None | Input object class |
        | y_class | str | None | Target object class |
        | get_items | Mapping[str, str] | None | Column remapping dictionary for item extraction|
        | base_path | str | None | Base path for relative files |
        | folders | Mapping[str, str] | None | Folder mapping configuration |
        | suffixes | Mapping[str, str] | None | File suffix mapping |
        | keep_original | bool | False | Preserve original samples |
        | get_x | callable | None | Custom input extractor |
        | get_y | callable | None | Custom target extractor |
        | item_transforms | Sequence[callable] | None | Training item transforms |
        | val_item_transforms | Sequence[callable] | None | Validation item transforms |
        | transforms | Sequence[callable] | None | Training transforms |
        | val_transforms | Sequence[callable] | None | Validation transforms |
        | splitter | callable | None | Dataset splitting function |
        | valid_fraction | float | 0.2 | Validation split fraction |
        | seed | int | None | Random seed |
        | shuffle | bool | True | Shuffle training data |
        | batch_size | int | 64 | Batch size |
        | num_workers | int | 0 | Number of dataloader workers |
        | device | str | None | Device override |
        | drop_last | bool | False | Drop incomplete last batch |
        | pin_memory | bool | False | Pin memory in DataLoader |
        | persistent_workers | bool | False | Keep workers persistent |
        | show_summary | bool | False | Display dataset summary |
        | kwargs | dict | {} | Additional pipeline configuration |

        Returns
        -------
        DataLoaders
        """

        return cls._run_pipeline(
            data,
            task=task,
            dataset=dataset,
            backend=backend,
            val_data=val_data,
            x_keys=x_keys,
            y_keys=y_keys,
            x_class=x_class,
            y_class=y_class,
            get_items=get_items,
            base_path=base_path,
            folders=folders,
            suffixes=suffixes,
            keep_original=keep_original,
            get_x=get_x,
            get_y=get_y,
            item_transforms=item_transforms,
            val_item_transforms=val_item_transforms,
            transforms=transforms,
            val_transforms=val_transforms,
            splitter=splitter,
            valid_fraction=valid_fraction,
            seed=seed,
            shuffle=shuffle,
            batch_size=batch_size,
            num_workers=num_workers,
            device=device,
            drop_last=drop_last,
            pin_memory=pin_memory,
            persistent_workers=persistent_workers,
            show_summary=show_summary,
            **kwargs,  
        )

    # --------------------------------------------------
    @classmethod
    def create_from_yaml(
        cls,
        yaml_path: str,                               # YAML configuration file path
    ):
        """
        Create training and validation DataLoaders from YAML configuration.

        | Parameter | Type | Default | Description |
        |----------|------|----------|-------------|
        | yaml_path | str | — | YAML configuration file path |

        Returns
        -------
        DataLoaders
        """
        config = read_yaml(yaml_path) or {}
        config = {key: (None if value == "None" else value) for key, value in config.items()}

        data = config.pop("data", None)
        if data is None:
            raise ValueError("YAML config must contain a 'data' key for BioDataLoaders.create_from_yaml.")

        val_data = config.pop("val_data", None)
        task = config.pop("task", None)
        dataset = config.pop("dataset", None)
        backend = config.pop("backend", None)

        return cls._run_pipeline(
            data,
            val_data=val_data,
            task=task,
            dataset=dataset,
            backend=backend,
            **config,
        )
    
    # --------------------------------------------------
    @classmethod
    def test_dl(
        cls,
        data: Any,                                    # Test data source
        task: Optional[str] = None,                  # Registered task name
        dataset: Optional[str] = None,               # Dataset builder name
        backend: Optional[str] = None,               # Backend override
        **kwargs,                                    # Additional pipeline kwargs
    ):
        """
        Create a test DataLoader.

        Validation transforms are automatically applied and dataset
        splitting is disabled.

        | Parameter | Type | Default | Description |
        |----------|------|----------|-------------|
        | data | Any | — | Test data source |
        | task | str | None | Registered task name |
        | dataset | str | None | Dataset builder name |
        | backend | str | None | Backend override |
        | kwargs | dict | {} | Additional pipeline configuration |

        Returns
        -------
        DataLoader
        """

        return cls._run_pipeline(
            data,
            task=task,
            dataset=dataset,
            backend=backend,
            mode="test",
            **kwargs
        )
    
    # --------------------------------------------------
    @classmethod
    def test_dl_from_yaml(
        cls,
        yaml_path: str,                               # YAML configuration file path
    ):
        """
        Create a test DataLoader from YAML configuration.

        | Parameter | Type | Default | Description |
        |----------|------|----------|-------------|
        | yaml_path | str | — | YAML configuration file path |

        Returns
        -------
        DataLoader
        """
        
        config = read_yaml(yaml_path) or {}
        config = {key: (None if value == "None" else value) for key, value in config.items()}

        data = config.pop("data", None)
        if data is None:
            raise ValueError("YAML config must contain a 'data' key for BioDataLoaders.test_dl_from_yaml.")

        task = config.pop("task", None)
        dataset = config.pop("dataset", None)
        backend = config.pop("backend", None)

        return cls.test_dl(
            data,
            task=task,
            dataset=dataset,
            backend=backend,
            **config,
        )

In [ ]:
show_doc(BioDataLoaders.create)

---

[source](https://github.com/deepCLEM/bioMONAI/blob/main/bioMONAI/data/load.py#L1658){target="_blank" style="float:right; font-size:smaller"}

### BioDataLoaders.create

```python

def create(
    data:Any, # Training data source
    task:Optional=None, # Registered task name
    dataset:Optional=None, # Dataset builder name
    backend:Optional=None, # Backend override
    val_data:Optional=None, # Optional validation data
    x_keys:Optional=None, # Input column keys
    y_keys:Optional=None, # Target column keys
    x_class:Optional=None, # Input object/type class
    y_class:Optional=None, # Target object/type class
    get_items:Optional=None, # Column remapping dictionary
    base_path:Optional=None, # Base path for relative files
    folders:Optional=None, # Folder mapping
    suffixes:Optional=None, # File suffix mapping
    keep_original:bool=False, # Preserve original samples
    get_x:Optional=None, # Custom input extractor
    get_y:Optional=None, # Custom target extractor
    item_transforms:Optional=None, # Training item transforms
    val_item_transforms:Optional=None, # Validation item transforms
    transforms:Optional=None, # Training transforms
    val_transforms:Optional=None, # Validation transforms
    splitter:Optional=None, # Dataset splitter
    valid_fraction:float=0.2, # Validation split fraction
    seed:Optional=None, # Random seed
    shuffle:bool=True, # Shuffle training data
    batch_size:int=64, # Batch size
    num_workers:int=0, # Number of workers
    device:Optional=None, # Device override
    drop_last:bool=False, # Drop incomplete last batch
    pin_memory:bool=False, # Pin memory in DataLoader
    persistent_workers:bool=False, # Keep workers persistent
    show_summary:bool=False, # Display dataset summary
    kwargs:VAR_KEYWORD
):


```

*Create training and validation DataLoaders.*

| Parameter | Type | Default | Description |
|----------|------|----------|-------------|
| data | Any | — | Training data source |
| task | str | None | Registered task name |
| dataset | str | None | Dataset builder name |
| backend | str | None | Backend override |
| val_data | Any | None | Optional validation dataset |
| x_keys | Sequence[str] | None | Input column keys |
| y_keys | Sequence[str] | None | Target column keys |
| x_class | str | None | Input object class |
| y_class | str | None | Target object class |
| get_items | Mapping[str, str] | None | Column remapping dictionary for item extraction|
| base_path | str | None | Base path for relative files |
| folders | Mapping[str, str] | None | Folder mapping configuration |
| suffixes | Mapping[str, str] | None | File suffix mapping |
| keep_original | bool | False | Preserve original samples |
| get_x | callable | None | Custom input extractor |
| get_y | callable | None | Custom target extractor |
| item_transforms | Sequence[callable] | None | Training item transforms |
| val_item_transforms | Sequence[callable] | None | Validation item transforms |
| transforms | Sequence[callable] | None | Training transforms |
| val_transforms | Sequence[callable] | None | Validation transforms |
| splitter | callable | None | Dataset splitting function |
| valid_fraction | float | 0.2 | Validation split fraction |
| seed | int | None | Random seed |
| shuffle | bool | True | Shuffle training data |
| batch_size | int | 64 | Batch size |
| num_workers | int | 0 | Number of dataloader workers |
| device | str | None | Device override |
| drop_last | bool | False | Drop incomplete last batch |
| pin_memory | bool | False | Pin memory in DataLoader |
| persistent_workers | bool | False | Keep workers persistent |
| show_summary | bool | False | Display dataset summary |
| kwargs | dict | {} | Additional pipeline configuration |

In [ ]:
show_doc(BioDataLoaders.create_from_yaml)

---

[source](https://github.com/deepCLEM/bioMONAI/blob/main/bioMONAI/data/load.py#L1787){target="_blank" style="float:right; font-size:smaller"}

### BioDataLoaders.create_from_yaml

```python

def create_from_yaml(
    yaml_path:str, # YAML configuration file path
):


```

*Create training and validation DataLoaders from YAML configuration.*

| Parameter | Type | Default | Description |
|----------|------|----------|-------------|
| yaml_path | str | — | YAML configuration file path |

In [ ]:
show_doc(BioDataLoaders.test_dl)

---

[source](https://github.com/deepCLEM/bioMONAI/blob/main/bioMONAI/data/load.py#L1825){target="_blank" style="float:right; font-size:smaller"}

### BioDataLoaders.test_dl

```python

def test_dl(
    data:Any, # Test data source
    task:Optional=None, # Registered task name
    dataset:Optional=None, # Dataset builder name
    backend:Optional=None, # Backend override
    kwargs:VAR_KEYWORD
):


```

*Create a test DataLoader.*

Validation transforms are automatically applied and dataset
splitting is disabled.

| Parameter | Type | Default | Description |
|----------|------|----------|-------------|
| data | Any | — | Test data source |
| task | str | None | Registered task name |
| dataset | str | None | Dataset builder name |
| backend | str | None | Backend override |
| kwargs | dict | {} | Additional pipeline configuration |

In [ ]:
import tempfile
import yaml

# Example: The Grand Orchestrator (BioDataLoaders)
# BioDataLoaders provides a unified, single-entry API to build your entire pipeline.
# You can create DataLoaders directly via Python, or by loading a YAML configuration file.

# --- Method 1: Using the Python API ---
dls_python = BioDataLoaders.create(
    data=[{"image": "img1.nii", "label": 1}, {"image": "img2.nii", "label": 0}],
    task="classification",  # Automatically applies ClassificationTask defaults
    dataset="cache",        # Uses the CacheDatasetBuilder
    backend="monai",        # Wraps it in a MonaiLoader
    batch_size=8,
    val_batch_size=4
)
print("Pipeline configured via Python API successfully.")

# --- Method 2: Using a YAML Configuration File ---
yaml_content = {
    "data": "path/to/my/dataset.csv",
    "task": "classification",
    "dataset": "datablock",
    "backend": "fastai",
    "batch_size": 32
}

with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    yaml.dump(yaml_content, f)
    yaml_path = f.name

try:
    dls_yaml = BioDataLoaders.create_from_yaml(yaml_path)
    print("Pipeline configured via YAML successfully.")
except Exception as e:
    pass # Dummy example

Pipeline configured via Python API successfully.


In [ ]:
#| hide
import os
import tempfile
import yaml
from unittest.mock import patch
from fastcore.test import test_eq

# Verify BioDataLoaders internal methods and public API

def test_load_dataframe():
    class DummySource:
        def __init__(self, data, **kwargs): self.data = data
        def load(self): return list(self.data)

    SOURCE_REGISTRY["dummy"] = DummySource
    orig_detect = globals().get("detect_source")
    globals()["detect_source"] = lambda data: "dummy"

    try:
        train_data = [{"image": "a", "label": 0}, {"image": "b", "label": 1}]
        val_data = [{"image": "c", "label": 0}]
        
        ctx = PipelineContext(data=train_data, val_data=val_data, config={"get_items": {}})
        ctx = BioDataLoaders._load_data(ctx)
        data = ctx.records

        test_eq(len(data), 3)
        test_eq(ColSplitter()(data), ([0, 1], [2]))
        for d in data: test_eq("is_valid" in d, True)
    finally:
        if orig_detect: globals()["detect_source"] = orig_detect
        else: del globals()["detect_source"]

def test_build_dataset_internals():
    data = [{"image": "a", "label": 0}, {"image": "b", "label": 1}]
    ctx = PipelineContext(records=data, dataset_name="dataset", mode="train")
    
    ctx = BioDataLoaders._build_dataset(ctx)
    # Accept both the real MONAI Dataset and our Mock version
    test_eq(type(ctx.train_ds).__name__ in ['Dataset', 'MonaiDataset', 'MockMonaiDataset'], True)
    test_eq(type(ctx.valid_ds).__name__ in ['Dataset', 'MonaiDataset', 'MockMonaiDataset'], True)
    test_eq(ctx.backend, 'monai')

    ctx.dataset_name = "cache"
    ctx.backend = None
    ctx = BioDataLoaders._build_dataset(ctx)
    # Accept both the real CacheDataset and our Mock version
    test_eq(type(ctx.train_ds).__name__ in ['CacheDataset', 'MockCacheDataset'], True)
    test_eq(type(ctx.valid_ds).__name__ in ['CacheDataset', 'MockCacheDataset'], True)
    test_eq(ctx.backend, 'monai')

    ctx.dataset_name = "datablock"
    ctx.backend = None
    ctx = BioDataLoaders._build_dataset(ctx)
    test_eq(type(ctx.train_ds).__name__, 'DataBlock')
    test_eq(type(ctx.valid_ds).__name__, 'list')
    test_eq(ctx.backend, 'fastai')

def test_biodataloaders_api():
    with patch.object(BioDataLoaders, '_run_pipeline', return_value="MockDLS") as mock_run:
        dls = BioDataLoaders.create(data=[1, 2], batch_size=16, backend="fastai", task="classification")
        test_eq(dls, "MockDLS")
        test_eq(mock_run.call_args[0][0], [1, 2])
        test_eq(mock_run.call_args[1]["batch_size"], 16)

        with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.yaml') as f:
            yaml.dump({"data": "dummy.csv", "batch_size": 64}, f)
            temp_path = f.name
            
        try:
            yaml_dls = BioDataLoaders.create_from_yaml(temp_path)
            test_eq(yaml_dls, "MockDLS")
            test_eq(mock_run.call_args[0][0], "dummy.csv")
            test_eq(mock_run.call_args[1]["batch_size"], 64)
        finally:
            os.remove(temp_path)

# Run all merged tests
test_load_dataframe()
test_build_dataset_internals()
test_biodataloaders_api()
print("All BioDataLoaders internal and API tests passed")

All BioDataLoaders internal and API tests passed


### BioDataLoaders: Specialized Factory Methods

The `BioDataLoaders` module provides highly convenient, fastai-style factory methods to streamline the preparation of datasets for machine learning models. Built on top of fastai’s `DataLoaders`, it wraps various data ingestion strategies and automatically utilizes `BioImageBlock` as the foundational `TransformBlock` to handle medical and biological imaging tensors appropriately.

These specialized methods are designed to build your entire data pipeline—from raw files to batched tensors—in a single, expressive setup, depending entirely on how your raw data is structured.

#### Core & Declarative Builders
* **`from_source`**: The foundational factory method. It creates and returns a `DataLoader` by initializing a `BioDataBlock` using explicitly provided keyword arguments (e.g., `blocks`, `splitter`, `get_x`, `get_y`, `item_tfms`).
* **`from_yaml`**: A declarative builder that reads a YAML configuration file, parses the parameters, maps them to the correct dataloader and datablock operations, and fully instantiates the pipeline.

#### Generic Dataset Builders
These methods are ideal for general tasks (such as image-to-image translation, segmentation, or regression) where inputs and targets might both be complex objects.
* **`from_folder`**: Constructs loaders directly from a directory structure, typically relying on `train` and `valid` subfolders or a defined validation percentage.
* **`from_df`**: Builds loaders from a Pandas DataFrame, using specified column indices or names to locate input files (`fn_col`) and target data (`target_col`).
* **`from_csv`**: A convenience wrapper that dynamically reads a CSV file into a DataFrame and delegates the processing to `from_df`.

#### Classification Dataset Builders
These methods are specifically tailored for image classification workflows, automatically configuring a `CategoryBlock` (or `MultiCategoryBlock` for multi-label tasks) for the targets.
* **`class_from_folder`**: Infers class labels automatically from the names of the parent directories.
* **`class_from_df` / `class_from_csv`**: Extracts image paths and categorical labels directly from tabular data structures.
* **`class_from_path_func` / `class_from_path_re`**: Derives categorical labels dynamically from file paths using a custom Python mapping function or a precise Regular Expression pattern.
* **`class_from_lists`**: Constructs loaders directly from raw, paired Python lists of filenames and their corresponding discrete labels.

In [ ]:
#| export

# =================================================================
# WE CREATE THE MISSING BLOCK SO THE FACTORY METHODS CAN USE IT
# =================================================================
def BioImageBlock(cls=BioImage):
    """A fastai block specifically for loading BioImages."""
    # Assuming BioImage has a .create method. If not, TransformBlock handles it.
    return TransformBlock(type_tfms=getattr(cls, 'create', cls))

def from_source(cls, 
                data_source, # The source of the data to be loaded by the dataloader. This can be any type that is compatible with the dataloading method specified in kwargs (e.g., paths, datasets).
                show_summary:bool=False, # If True, print a summary of the BioDataBlock after creation.
                **kwargs, # Additional keyword arguments to configure the DataLoader and BioDataBlock. Supported keys include: 'blocks', 'dl_type', 'get_items', 'get_y', 'get_x', 'getters', 'n_inp', 'item_tfms', 'batch_tfms'.
                ):
    """
    Create and return a DataLoader from a BioDataBlock using provided keyword arguments.
    
    Returns a  DataLoader: A PyTorch DataLoader object populated with the data from the BioDataBlock.
                    If show_summary is True, it also prints a summary of the datablock after creation.
    
    """
    # Define the keys for BioDataBlock operations
    datablock_ops_keys = ['blocks','dl_type','get_items','get_y','get_x','getters','n_inp','item_tfms','batch_tfms','splitter']
    
    # Filter and assign kwargs to datablock_ops dictionary for BioDataBlock initialization
    datablock_ops = {key: value for key, value in kwargs.items() if key in datablock_ops_keys}
    
    # Filter and assign remaining kwargs to dataloader_ops dictionary for DataLoader creation
    dataloader_ops = {key: value for key, value in kwargs.items() if key not in datablock_ops_keys}
    
    # Initialize BioDataBlock with specified operations
    datablock = BioDataBlock(**datablock_ops)

    # Create and return the DataLoader from the initialized BioDataBlock
    dataloder = datablock.dataloaders(data_source, **dataloader_ops)
    
    # Optionally print a summary of the BioDataBlock if show_summary is True
    if show_summary:
        # bs = dataloader_ops['bs'] if dataloader_ops['bs'] is not None else 1
        # securely get batch size with a default of 1 if not specified
        bs = dataloader_ops.get('bs', 1)
        print(datablock.summary(data_source, bs=bs))
    
    return dataloder


def from_folder(cls, path, get_target_fn, train='train', valid='valid', valid_pct=None, seed=None, item_tfms=None,
                batch_tfms=None, img_cls=BioImage, target_img_cls=BioImage, get_items=None, **kwargs):
    "Create from dataset in `path` with `train` and `valid` subfolders (or provide `valid_pct`)"
    splitter = GrandparentSplitter(train_name=train, valid_name=valid) if valid_pct is None else RandomSplitter(valid_pct, seed=seed)
    if get_items is None:
        get_items = get_image_files if valid_pct else partial(get_image_files, folders=[train, valid])
    ops = { 
        'blocks':       (BioImageBlock(img_cls), BioImageBlock(target_img_cls)),
        'get_items':    get_items,
        'splitter':     splitter,
        'get_y':        get_target_fn,
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(path, **ops, **kwargs)


def from_df(cls, df, path='.', valid_pct=0.2, seed=None, fn_col=0, folder=None, pref=None, suff='', target_col=1, target_folder=None, target_suff='',
            valid_col=None, item_tfms=None, batch_tfms=None, img_cls=BioImage, target_img_cls=BioImage, **kwargs):
    "Create from `df` using `fn_col` and `target_col`"
    if pref is None:
        pref = f'{Path(path) if folder is None else Path(path)/folder}{os.path.sep}'
    if folder is None:
        target_pref = pref
    else:
        target_pref = f'{Path(path)/target_folder}{os.path.sep}'

    def _split(o):
        df = o if isinstance(o, pd.DataFrame) else pd.DataFrame(o)
        train, valid = (
            RandomSplitter(valid_pct, seed=seed)(df)
            if valid_col is None
            else ColSplitter(valid_col)(df)
        )
        return L(train), L(valid)

    splitter = _split     
    
    target_img_cls = img_cls if target_img_cls is None else target_img_cls
    ops = { 
        'blocks':       (BioImageBlock(img_cls), BioImageBlock(target_img_cls)),
        'get_items':    None,
        'splitter':     splitter,
        'get_x':        ColReader(fn_col, pref=pref, suff=suff),
        'get_y':        ColReader(target_col, pref=target_pref, suff=target_suff),
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(df, **ops, **kwargs)


def from_csv(cls, path, csv_fname='train.csv', header='infer', delimiter=None, quoting=0, **kwargs):
    "Create from `path/csv_fname` using `fn_col` and `target_col`"
    df = pd.read_csv(Path(path)/csv_fname, header=header, delimiter=delimiter, quoting=quoting)
    return cls.from_df(df, path=path, **kwargs)
    

def class_from_folder(cls, path, train='train', valid='valid', valid_pct=None, seed=None, vocab=None, item_tfms=None,
                batch_tfms=None, img_cls=BioImage, **kwargs):
    "Create from dataset in `path` with `train` and `valid` subfolders (or provide `valid_pct`)"
    splitter = GrandparentSplitter(train_name=train, valid_name=valid) if valid_pct is None else RandomSplitter(valid_pct, seed=seed)
    get_items = get_image_files if valid_pct else partial(get_image_files, folders=[train, valid])
    ops = { 
        'blocks':       (BioImageBlock(img_cls), CategoryBlock(vocab=vocab)),
        'get_items':    get_items,
        'splitter':     splitter,
        'get_y':        parent_label,
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(path, **ops, **kwargs)


def class_from_path_func(cls, path, fnames, label_func, valid_pct=0.2, seed=None, item_tfms=None, batch_tfms=None, 
                    img_cls=BioImage, **kwargs):
    "Create from list of `fnames` in `path`s with `label_func`"
    ops = { 
        'blocks':       (BioImageBlock(img_cls), CategoryBlock),
        'splitter':     RandomSplitter(valid_pct, seed=seed),
        'get_y':        label_func,
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(fnames, **ops, **kwargs)


def class_from_path_re(cls, path, fnames, pat, **kwargs):
    "Create from list of `fnames` in `path`s with re expression `pat`"
    return cls.class_from_path_func(path, fnames, RegexLabeller(pat), **kwargs)


def class_from_df(cls, df, path='.', valid_pct=0.2, seed=None, fn_col='filename', folder=None, suff='', label_col='label', label_delim=None,
            y_block=None, valid_col=None, item_tfms=None, batch_tfms=None, img_cls=BioImage, **kwargs):
    "Create from `df` using `fn_col` and `label_col`"
    pref = f'{Path(path) if folder is None else Path(path)/folder}{os.path.sep}'
    if y_block is None:
        is_multi = (is_listy(label_col) and len(label_col) > 1) or label_delim is not None
        y_block = MultiCategoryBlock if is_multi else CategoryBlock
    def _split(o):
        df = o if isinstance(o, pd.DataFrame) else pd.DataFrame(o)
        train, valid = (
            RandomSplitter(valid_pct, seed=seed)(df)
            if valid_col is None
            else ColSplitter(valid_col)(df)
        )
        return L(train), L(valid)
    splitter = _split      

    ops = { 
        'blocks':       (BioImageBlock(img_cls), y_block),
        'get_items':    None,
        'splitter':     splitter,
        'get_x':        ColReader(fn_col, pref=pref, suff=suff),
        'get_y':        ColReader(label_col, label_delim=label_delim),
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(df, **ops, **kwargs)


def class_from_csv(cls, path, csv_fname='labels.csv', header='infer', delimiter=None, quoting=0, **kwargs):
    "Create from `path/csv_fname` using `fn_col` and `label_col`"
    df = pd.read_csv(Path(path)/csv_fname, header=header, delimiter=delimiter, quoting=quoting)
    return cls.class_from_df(df, path=path, **kwargs)


def class_from_lists(cls, path, fnames, labels, valid_pct=0.2, seed:int=None, y_block=None, item_tfms=None, batch_tfms=None,
                img_cls=BioImage, **kwargs):
    "Create from list of `fnames` and `labels` in `path`"
    if y_block is None:
        y_block = MultiCategoryBlock if is_listy(labels[0]) and len(labels[0]) > 1 else (
            RegressionBlock if isinstance(labels[0], float) else CategoryBlock)
    ops = { 
        'blocks':       (BioImageBlock(img_cls), y_block),
        'splitter':     RandomSplitter(valid_pct, seed=seed),
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source((fnames, labels), **ops, **kwargs)


def from_yaml(cls, data_source, yaml_path, show_summary:bool=False):

    "Create from `yaml_path` where `yaml_path` is a yaml file"


    # Read the yaml file to obtain a dictionary with the configuration
    config = read_yaml(yaml_path)
    
    # Turn string Nones into Nonetype and remove keys where the value is set to Nonetype
    config = {key: (None if value == "None" else value) for key, value in config.items()}

    # DEFINE THE KEYS THAT ARE AVAILABLE FOR USE 
    # Define the keys used by fastTrainer
    fastrainer_ops_keys = ['loss_fn', 'optimizer', 'lr', 'splitter', 'callbacks', 'metrics', 'path', 'model_dir', 'wd', 
                            'wd_bn_bias', 'train_bn', 'moms', 'default_cbs']
    
    # Define the keys used by biodataloader
    biodataloader_ops_keys = ['bs', 'shuffle_train', 'shuffle', 'val_shuffle', 'n', 'path', 'dl_type', 'dl_kwargs', 'device', 
                                'drop_last', 'val_bs', 'num_workers', 'verbose', 'do_setup', 'pin_memory', 'timeout', 'batch_size', 
                                'indexed', 'persistent_workers', 'pin_memory_device', 'wif', 'before_iter', 'after_item', 'before_batch', 
                                'after_batch', 'after_iter', 'create_batches', 'create_item', 'create_batch', 'retain', 'get_idxs', 'sample', 
                                'shuffle_fn', 'do_batch']

    # Define the keys used by biodatablocks
    biodatablocks_ops_keys = ['blocks','dl_type','get_items','get_y','get_x','getters','n_inp','item_tfms','batch_tfms','splitter']


    # FILTER THE YAML FILE TO ONLY INCLUDE THE KEYS THE VALID KEYS
    biodatablock_ops = {key: value for key, value in config.items() if key in biodatablocks_ops_keys}
    biodataloader_ops = {key: value for key, value in config.items() if key in biodataloader_ops_keys}


    # Obtain and define default values for the splitter within the BioDataBlock
    train = config.get('train', 'train') 
    valid = config.get('valid', 'val')  
    valid_pct = config.get('valid_pct', None) 
    seed = config.get('seed', None) 
    

    # Initialize the splitter
    if valid_pct is not None:
        splitter = RandomSplitter(valid_pct, seed=seed)
        get_items = get_image_files  
    else:
        splitter = GrandparentSplitter(train_name=train, valid_name=valid)
        get_items = partial(get_image_files, folders=[train, valid])  

    # Turn item_tfms and batch_tfms into lists of functions 
    item_tfms = config.get('item_tfms', None)
    if item_tfms is not None:
        item_tfms = dictlist_to_funclist(item_tfms)

    batch_tfms = config.get('batch_tfms', None)
    if batch_tfms is not None:   
        batch_tfms = dictlist_to_funclist(batch_tfms)

    # Update biodatablock_ops with the splitter
    biodatablock_ops.update({
        "blocks": (BioImageBlock(cls=BioImage), CategoryBlock),
        "get_items": get_items,
        "splitter": splitter,
        "get_y": parent_label,
        "item_tfms": item_tfms,
        "batch_tfms": batch_tfms
    })

    biodatablock_ops = {key: value for key, value in biodatablock_ops.items() if value is not None}

    biodataloader_ops = {key: value for key, value in biodataloader_ops.items() if value is not None}
    
    # Create BioDataBlock
    datablock = BioDataBlock(**biodatablock_ops)

    # Optionally print a summary of the BioDataBlock if show_summary is True
    if show_summary:
        bs = biodataloader_ops['bs'] if biodataloader_ops['bs'] is not None else 1
        print(datablock.summary(data_source, bs=bs))


    # Unpack biodataloader_ops directly (including bs)
    dataloder = datablock.dataloaders(data_source, **biodataloader_ops)
    
    return dataloder


In [ ]:
#| export
BioDataLoaders.from_source = classmethod(from_source)
BioDataLoaders.from_folder = classmethod(from_folder)
BioDataLoaders.from_df = classmethod(from_df)
BioDataLoaders.from_csv = classmethod(from_csv)
BioDataLoaders.class_from_folder = classmethod(class_from_folder)
BioDataLoaders.class_from_path_func = classmethod(class_from_path_func)
BioDataLoaders.class_from_df = classmethod(class_from_df)
BioDataLoaders.class_from_csv = classmethod(class_from_csv)
BioDataLoaders.class_from_path_re = classmethod(class_from_path_re)
BioDataLoaders.class_from_lists = classmethod(class_from_lists)
BioDataLoaders.from_yaml = classmethod(from_yaml)


In [ ]:
#| export
BioDataLoaders.from_source = delegates(to=BioDataLoaders.from_dblock)(BioDataLoaders.from_source)
BioDataLoaders.from_folder = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.from_folder)
BioDataLoaders.from_df = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.from_df)
BioDataLoaders.from_csv = delegates(to=BioDataLoaders.from_df)(BioDataLoaders.from_csv)
BioDataLoaders.class_from_folder = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.class_from_folder)
BioDataLoaders.class_from_path_func = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.class_from_path_func)
BioDataLoaders.class_from_df = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.class_from_df)
BioDataLoaders.class_from_csv = delegates(to=BioDataLoaders.class_from_df)(BioDataLoaders.class_from_csv)
BioDataLoaders.class_from_path_re = delegates(to=BioDataLoaders.class_from_path_func)(BioDataLoaders.class_from_path_re)
BioDataLoaders.class_from_lists = delegates(to=BioDataLoaders.class_from_df)(BioDataLoaders.class_from_lists)

In [ ]:
import pandas as pd

# Example: Using Fastai-style Factory Methods
# BioDataLoaders provides incredibly convenient one-liners to build your 
# DataLoaders directly from DataFrames, CSVs, or Folder structures.

# --- 1. From a Pandas DataFrame ---
df = pd.DataFrame({
    "filename": ["scan_001.nii", "scan_002.nii"],
    "label": ["healthy", "tumor"],
    "is_valid": [False, True]
})

try:
    # Build a classification dataloader mapping the columns directly
    dls_df = BioDataLoaders.class_from_df(
        df,
        fn_col="filename",
        label_col="label",
        valid_col="is_valid",
        batch_size=8
    )
    print("DataLoaders built from DataFrame successfully!")
except Exception as e:
    # (Fails gracefully in documentation if the dummy files don't exist on disk)
    pass


# --- 2. From a Folder Structure ---
try:
    # Build a classification dataloader where the folder names are the labels
    # e.g., dataset/train/tumor/img1.nii
    dls_folder = BioDataLoaders.class_from_folder(
        path="path/to/dataset",
        train="train",
        valid="valid",
        batch_size=16
    )
    print("DataLoaders built from Folders successfully!")
except Exception as e:
    pass

In [ ]:
#| hide
from unittest.mock import patch
import pandas as pd
from fastcore.test import test_eq

# Verify that all factory methods correctly parse arguments and route to `from_source`

def test_factory_methods():
    dummy_df = pd.DataFrame({"filename": ["a", "b"], "label": [0, 1]})

    # --- Test 1: class_from_df routing ---
    with patch.object(BioDataLoaders, 'from_source', return_value="MockDLS") as mock_from_source:
        
        dls = BioDataLoaders.class_from_df(dummy_df, fn_col="filename", label_col="label")

        test_eq(dls, "MockDLS")
        passed_data = mock_from_source.call_args[0][0]
        test_eq(passed_data.equals(dummy_df), True)
        
        ops_kwargs = mock_from_source.call_args[1]
        test_eq('blocks' in ops_kwargs, True)
        test_eq('get_x' in ops_kwargs, True)
        test_eq('get_y' in ops_kwargs, True)

    # --- Test 2: class_from_csv routing ---
    with patch('pandas.read_csv', return_value=dummy_df) as mock_read_csv:
        with patch.object(BioDataLoaders, 'class_from_df', return_value="MockCSV_DLS") as mock_class_from_df:
            
            dls_csv = BioDataLoaders.class_from_csv("dummy_dir", "dummy.csv")
            test_eq(dls_csv, "MockCSV_DLS")
            test_eq(mock_read_csv.called, True)

    return "All factory method routing tests passed"

test_eq(test_factory_methods(), "All factory method routing tests passed")

### Loading MONAI Datasets

For users who prefer to build their datasets using MONAI's native ecosystem (such as `Dataset`, `CacheDataset`, or `PersistentDataset`), `BioDataLoaders` provides specialized integration methods. These factory methods act as a seamless bridge, taking your MONAI data structures and wrapping them into highly optimized fastai `DataLoaders` ready for the training loop.

* **`from_monai`**: This method takes your pre-built MONAI datasets and packages them. Crucially, it manages parameter routing automatically—scaling the validation batch size by default (e.g., `val_bs_factor=2`) and allowing you to override validation-specific arguments (like `num_workers` or `pin_memory`) simply by using the `val_` prefix.
* **`from_monai_ds`**: For an even more streamlined workflow, this method allows you to skip manually instantiating your MONAI datasets before creating your DataLoaders. You simply pass the uninstantiated dataset class alongside your raw data lists. It automatically instantiates both the training and validation datasets, intelligently merges your configuration dictionaries (`dataset_kwargs` and `val_dataset_kwargs`) to inherit baseline configurations, and seamlessly delegates the rest of the DataLoader construction to the standard `from_monai` method.

In [ ]:
#| export

def from_monai(
    cls,
    train_ds,            # MONAI training dataset
    val_ds=None,         # MONAI validation dataset
    x_keys="image",      # Key(s) used as model inputs
    y_keys="label",      # Key(s) used as targets
    bs=64,               # Training batch size
    val_bs=None,         # Validation batch size (overrides automatic scaling)
    val_bs_factor=2,     # Multiplier applied to bs when val_bs is None
    shuffle=True,        # Shuffle training dataset
    val_shuffle=False,   # Shuffle validation dataset (defaults to False)
    show_summary=False,  # Print basic dataloader summary
    vocab=None,          # Optional class names for classification tasks
    **dl_kwargs,         # Additional torch DataLoader kwargs (train + val_)
):
    """
    Create fastai-compatible `DataLoaders` from MONAI dictionary datasets.

    Parameters
    ----------
    train_ds : Dataset
        Training dataset (typically MONAI `Dataset`, `CacheDataset`, etc.)

    val_ds : Dataset, optional
        Validation dataset.

    x_keys : str or Sequence[str]
        Dictionary key(s) to extract as model inputs.

    y_keys : str or Sequence[str]
        Dictionary key(s) to extract as targets.

    bs : int
        Training batch size.

    val_bs : int, optional
        Validation batch size. If None, computed as `bs * val_bs_factor`.

    val_bs_factor : int
        Multiplier applied to training batch size for validation.

    shuffle : bool
        Whether to shuffle the training dataset.

    val_shuffle : bool
        Whether to shuffle the validation dataset. Defaults to False.

    show_summary : bool
        If True, prints number of batches in each dataloader.

    **dl_kwargs
        Additional arguments passed to `torch.utils.data.DataLoader`.

        Validation-specific arguments can be specified using the `val_`
        prefix.

        Example:

        - `num_workers=8`
        - `pin_memory=True`
        - `prefetch_factor=4`
        - `val_prefetch_factor=2`
    """

    # ---- wrap datasets ----
    train_ds = ReadDictDataset(train_ds, x_keys=x_keys, y_keys=y_keys)
    val_ds = ReadDictDataset(val_ds, x_keys=x_keys, y_keys=y_keys) if val_ds else None

    # ----patch datasets ----
    if vocab:
        train_ds = _patch_dataset(train_ds, vocab=vocab)
        val_ds = _patch_dataset(val_ds, vocab=vocab) if val_ds else None


    # ---- split train / val kwargs ----
    train_kwargs = {}
    val_kwargs = {}

    for k, v in dl_kwargs.items():
        if k.startswith("val_"):
            val_kwargs[k[4:]] = v
        else:
            train_kwargs[k] = v

    # ---- mirror train → val defaults ----
    for k, v in train_kwargs.items():
        val_kwargs.setdefault(k, v)

    # ---- enforce sensible validation defaults ----
    val_kwargs.setdefault("shuffle", val_shuffle)
    val_kwargs.setdefault("drop_last", False)

    # ---- base train args ----
    train_args = dict(
        dataset=train_ds,
        batch_size=bs,
        shuffle=shuffle,
    )
    train_args.update(train_kwargs)

    train_dl = torchDataLoader(**train_args)

    # ---- validation batch size scaling ----
    val_dl = None
    if val_ds is not None:

        if val_bs is None:
            val_bs = bs * val_bs_factor

        val_args = dict(
            dataset=val_ds,
            batch_size=val_bs,
        )
        val_args.update(val_kwargs)

        val_dl = torchDataLoader(**val_args)

    # ---- patch fastai compatibility ----
    train_dl = _patch_dataloader(train_dl)
    if val_dl is not None:
        val_dl = _patch_dataloader(val_dl)

    # ---- build fastai DataLoaders ----
    dls = cls(train_dl, val_dl)

    if show_summary:
        _show_summary(train_dl, val_dl)

    return dls

BioDataLoaders.from_monai = classmethod(from_monai)

In [ ]:
# Example: Direct MONAI Dataset Integration
# You can seamlessly wrap standard MONAI dictionary datasets (like CacheDataset) 
# into fastai DataLoaders. The method automatically scales your validation batch size 
# and routes specific arguments to the train or validation loaders.

# 1. Create dummy MONAI-style data (lists of dictionaries)
dummy_train_ds = [{"image": "img1.nii", "label": 1}, {"image": "img2.nii", "label": 0}]
dummy_val_ds = [{"image": "img3.nii", "label": 1}]

try:
    # 2. Build the DataLoaders
    # Notice how we only specify 'bs=2'. Validation batch size automatically scales (2 * 2 = 4).
    # We also route 'num_workers=4' to training, but override it for validation with 'val_num_workers=1'.
    dls = BioDataLoaders.from_monai(
        train_ds=dummy_train_ds,
        val_ds=dummy_val_ds,
        bs=2,
        val_bs_factor=2,
        num_workers=4,
        val_num_workers=1,
        pin_memory=True # Applied to both train and validation
    )
    print("MONAI DataLoaders built successfully!")
except Exception as e:
    # Fails gracefully in the documentation if real internal torch/MONAI classes are missing
    pass

MONAI DataLoaders built successfully!


In [ ]:
#| hide
from unittest.mock import patch
from fastcore.test import test_eq

# 🛡️ ANTI-NAME-ERROR SHIELD & ROUTING VERIFICATION:
# We safely mock the internal dataset readers and torch dataloaders so we can 
# test the kwargs routing logic (like val_ prefixes and batch size scaling) 
# without triggering real PyTorch memory allocations or missing dependencies.

def test_from_monai_routing():
    dummy_train = [1, 2]
    dummy_val = [3]

    # Target globals that `from_monai` uses internally
    target_funcs = ['ReadDictDataset', 'torchDataLoader', '_patch_dataset', '_patch_dataloader', '_show_summary']
    orig_globals = {f: globals().get(f) for f in target_funcs}

    # Inject safe mock functions
    globals()['ReadDictDataset'] = lambda ds, **kwargs: ds
    globals()['_patch_dataset'] = lambda ds, **kwargs: ds
    globals()['_patch_dataloader'] = lambda dl, **kwargs: dl
    globals()['_show_summary'] = lambda t, v: None

    # Capture arguments passed to torchDataLoader
    captured_kwargs = []
    def mock_torchDataLoader(**kwargs):
        captured_kwargs.append(kwargs)
        return "MockDL"
    
    globals()['torchDataLoader'] = mock_torchDataLoader

    try:
        # Prevent the real BioDataLoaders.__init__ from firing with our fake string DataLoaders
        with patch.object(BioDataLoaders, '__init__', return_value=None):
            
            dls = BioDataLoaders.from_monai(
                dummy_train,
                val_ds=dummy_val,
                bs=8,
                val_bs_factor=2,
                num_workers=4,
                val_num_workers=2,
                pin_memory=True # Should inherit to val
            )

            # 1. Verify Training Kwargs
            train_kwargs = captured_kwargs[0]
            test_eq(train_kwargs['batch_size'], 8)
            test_eq(train_kwargs['num_workers'], 4)
            test_eq(train_kwargs['pin_memory'], True)
            test_eq(train_kwargs['shuffle'], True)

            # 2. Verify Validation Kwargs
            val_kwargs = captured_kwargs[1]
            test_eq(val_kwargs['batch_size'], 16) # Automatically scaled (8 * 2)
            test_eq(val_kwargs['num_workers'], 2) # Explicitly overridden
            test_eq(val_kwargs['pin_memory'], True) # Inherited from train
            test_eq(val_kwargs['shuffle'], False) # Default validation behavior

    finally:
        # Restore the original global environment perfectly
        for f in target_funcs:
            if orig_globals[f] is not None:
                globals()[f] = orig_globals[f]
            else:
                globals().pop(f, None)

    return "from_monai routing tests passed"

test_eq(test_from_monai_routing(), "from_monai routing tests passed")

In [ ]:
#| export
def from_monai_ds(
    cls,
    dataset_cls,                 # MONAI dataset class (Dataset, CacheDataset, etc.)
    train_data,                  # Training datalist
    train_transform=None,        # Training transform pipeline
    val_data=None,               # Optional validation datalist
    val_transform=None,          # Validation transforms
    dataset_kwargs=None,         # Extra args for dataset constructor
    val_dataset_kwargs=None,     # Validation dataset overrides
    **dl_kwargs                  # Passed to `from_monai`
):
    """
    Build `BioDataLoaders` from any MONAI dataset class.
    """

    dataset_kwargs = dataset_kwargs or {}
    val_dataset_kwargs = val_dataset_kwargs or {}

    # ---- training dataset ----
    train_ds = dataset_cls(
        train_data,
        transform=train_transform,
        **dataset_kwargs
    )

    # ---- validation dataset ----
    val_ds = None
    if val_data is not None:
        val_ds = dataset_cls(
            val_data,
            transform=val_transform,
            **{**dataset_kwargs, **val_dataset_kwargs}
        )

    # ---- delegate to main loader ----
    return cls.from_monai(
        train_ds=train_ds,
        val_ds=val_ds,
        **dl_kwargs
    )

BioDataLoaders.from_monai_ds = classmethod(from_monai_ds)

In [ ]:
# Example: Building DataLoaders directly from MONAI Dataset classes
# Instead of instantiating the datasets manually, you can pass the dataset class 
# (like MONAI's CacheDataset or PersistentDataset) along with your data lists.
# BioDataLoaders will instantiate them and pipe them into the dataloader engine.

# 1. Define dummy data lists
train_data = [{"image": "img1.nii", "label": 1}, {"image": "img2.nii", "label": 0}]
val_data = [{"image": "img3.nii", "label": 1}]

# 2. Define a mock MONAI Dataset class for this example
class MockMonaiDataset:
    def __init__(self, data, transform=None, **kwargs):
        self.data = data

try:
    # 3. Build the DataLoaders in one step
    dls = BioDataLoaders.from_monai_ds(
        dataset_cls=MockMonaiDataset,   # Pass the class itself
        train_data=train_data,
        val_data=val_data,
        dataset_kwargs={"cache_num": 10}, # Example: passing kwargs to the dataset constructor
        bs=4,                             # These arguments are routed seamlessly to `from_monai`
        val_bs_factor=2,
        num_workers=2
    )
    print("BioDataLoaders built from Dataset class successfully!")
except Exception as e:
    # Fails gracefully in the documentation if real internal torch/MONAI classes are missing
    pass

In [ ]:
#| hide
from unittest.mock import patch
from fastcore.test import test_eq

# Verify that `from_monai_ds` correctly instantiates the datasets, merges 
# the dictionary arguments properly, and delegates to `from_monai`.

def test_from_monai_ds_routing():
    dummy_train = [1, 2]
    dummy_val = [3]

    # Create a dummy class to inspect how it gets initialized
    class MockDataset:
        def __init__(self, data, transform=None, **kwargs):
            self.data = data
            self.transform = transform
            self.kwargs = kwargs

    # We patch `from_monai` to intercept the final call without triggering PyTorch
    with patch.object(BioDataLoaders, 'from_monai', return_value="MockDLS") as mock_from_monai:
        
        dls = BioDataLoaders.from_monai_ds(
            dataset_cls=MockDataset,
            train_data=dummy_train,
            train_transform="train_tfms",
            val_data=dummy_val,
            val_transform="val_tfms",
            dataset_kwargs={"base_cache": True},
            val_dataset_kwargs={"val_override": True}, # Should merge with dataset_kwargs
            bs=16, # Should route to from_monai
            pin_memory=True
        )

        test_eq(dls, "MockDLS")
        test_eq(mock_from_monai.called, True)
        
        # Extract the arguments passed down to from_monai
        call_kwargs = mock_from_monai.call_args[1]
        train_ds = call_kwargs['train_ds']
        val_ds = call_kwargs['val_ds']
        
        # 1. Verify Train Dataset initialization
        test_eq(train_ds.data, dummy_train)
        test_eq(train_ds.transform, "train_tfms")
        test_eq(train_ds.kwargs['base_cache'], True)
        
        # 2. Verify Validation Dataset initialization and dictionary merging
        test_eq(val_ds.data, dummy_val)
        test_eq(val_ds.transform, "val_tfms")
        test_eq(val_ds.kwargs['base_cache'], True)    # Inherited from base kwargs
        test_eq(val_ds.kwargs['val_override'], True)  # Added by val overrides

        # 3. Verify standard DataLoaders kwargs passed through
        test_eq(call_kwargs['bs'], 16)
        test_eq(call_kwargs['pin_memory'], True)

    return "from_monai_ds routing tests passed"

test_eq(test_from_monai_ds_routing(), "from_monai_ds routing tests passed")

### Test Datasets & Inference

This section provides robust utilities for generating fastai-compatible testing and inference `DataLoaders`. These tools are designed to effortlessly bridge various raw data formats and underlying dataset engines, ensuring your test environments perfectly inherit the configurations of your validation setup without manual boilerplate.

#### `_create_test_dl`
An internal utility that provides an automatic way to generate a testing `DataLoader` using a pre-existing `DataLoaders` object as a template. Rather than forcing you to redefine batch sizes, worker counts, or input keys, it intelligently inspects the validation dataloader inside your existing setup. It inherits those settings, automatically wraps your raw MONAI test dataset, applies the correct vocabulary mapping (if dealing with classification), and ensures that testing-specific flags (like `shuffle=False` and `drop_last=False`) are strictly enforced.

#### `test_biodataloader`
A versatile, universal routing utility designed to generate test/inference DataLoaders from almost any raw data source. Instead of requiring precisely formatted data, this function accepts a wide array of inputs: a Pandas `DataFrame`, a string path pointing to a `.csv` file, a directory path containing raw images, or an uninstantiated MONAI `Dataset`. It intelligently detects the input type, parses the data, and automatically delegates the construction process to either fastai's native `test_dl` method or our custom `_create_test_dl` MONAI wrapper.

In [ ]:
#| export
def _create_test_dl(
    test_ds,
    data,                 # existing fastai DataLoaders
    x_keys=None,
    y_keys=None,
    vocab=None,
    **dl_kwargs           # overrides for dataloader settings
):
    """
    Create a test DataLoader using validation DataLoader settings as defaults.

    Parameters
    ----------
    test_ds : Dataset
        MONAI test dataset.

    data : DataLoaders
        Existing fastai DataLoaders containing train/valid loaders.

    x_keys : str or sequence, optional
        Keys used as model inputs. Defaults to `data.valid.x_keys`.

    y_keys : str or sequence, optional
        Keys used as targets. Defaults to `data.valid.y_keys`.

    vocab : optional
        Vocabulary for classification tasks. Defaults to data.vocab if available.

    **dl_kwargs
        Explicit overrides for DataLoader parameters.

    Returns
    -------
    torch.utils.data.DataLoader
    """

    valid_dl = data.valid

    # ---- default keys from validation dataloader ----
    if x_keys is None:
        x_keys = getattr(valid_dl, "x_keys", "image")

    if y_keys is None:
        y_keys = getattr(valid_dl, "y_keys", "label")

    # ---- wrap dataset ----
    test_ds = ReadDictDataset(test_ds, x_keys=x_keys, y_keys=y_keys)

    # ---- vocab fallback ----
    if vocab is None and hasattr(data, "vocab"):
        vocab = data.vocab

    if vocab:
        test_ds = _patch_dataset(test_ds, vocab=vocab)

    # ---- copy validation dataloader settings ----
    base_kwargs = {
        "batch_size": valid_dl.batch_size,
        "num_workers": valid_dl.num_workers,
        "pin_memory": getattr(valid_dl, "pin_memory", False),
        "drop_last": False,
        "shuffle": False,
    }

    # optional attributes if present
    for attr in ["prefetch_factor", "persistent_workers"]:
        if hasattr(valid_dl, attr):
            base_kwargs[attr] = getattr(valid_dl, attr)

    # ---- allow explicit overrides ----
    base_kwargs.update(dl_kwargs)

    # ---- build dataloader ----
    test_dl = torchDataLoader(
        dataset=test_ds,
        **base_kwargs
    )

    # ---- fastai compatibility patch ----
    test_dl = _patch_dataloader(test_dl)

    return test_dl

In [ ]:
#| hide
from unittest.mock import patch
from types import SimpleNamespace
from fastcore.test import test_eq

# 🛡️ ANTI-NAME-ERROR SHIELD & ROUTING VERIFICATION:
# We safely mock the internal dataset readers and torch dataloaders so we can 
# test the inheritance logic without triggering real PyTorch memory allocations.

def test_create_test_dl_routing():
    # Target globals that `_create_test_dl` uses internally
    target_funcs = ['ReadDictDataset', 'torchDataLoader', '_patch_dataset', '_patch_dataloader']
    orig_globals = {f: globals().get(f) for f in target_funcs}

    # Inject safe mock classes and functions
    class MockReadDictDataset:
        def __init__(self, ds, x_keys, y_keys):
            self.ds = ds
            self.x_keys = x_keys
            self.y_keys = y_keys
            
    globals()['ReadDictDataset'] = MockReadDictDataset
    globals()['_patch_dataset'] = lambda ds, vocab: setattr(ds, 'vocab', vocab) or ds
    globals()['_patch_dataloader'] = lambda dl: dl

    # Capture arguments passed to torchDataLoader
    captured_kwargs = {}
    def mock_torchDataLoader(dataset, **kwargs):
        captured_kwargs.clear()
        captured_kwargs.update(kwargs)
        captured_kwargs['dataset'] = dataset
        return "MockTestDL"
    
    globals()['torchDataLoader'] = mock_torchDataLoader

    try:
        # Create fake objects to act as the existing fastai DataLoaders
        mock_valid_dl = SimpleNamespace(
            batch_size=32, 
            num_workers=4, 
            pin_memory=True, 
            x_keys="scan", 
            y_keys="mask",
            prefetch_factor=2
        )
        mock_dls = SimpleNamespace(valid=mock_valid_dl, vocab=["A", "B"])
        dummy_test_ds = [1, 2, 3]

        # --- 1. Test standard inheritance ---
        test_dl = _create_test_dl(dummy_test_ds, mock_dls)
        
        test_eq(test_dl, "MockTestDL")
        
        # Verify kwargs inherited from valid_dl
        test_eq(captured_kwargs['batch_size'], 32)
        test_eq(captured_kwargs['num_workers'], 4)
        test_eq(captured_kwargs['pin_memory'], True)
        test_eq(captured_kwargs['prefetch_factor'], 2)
        
        # Verify forced validation settings
        test_eq(captured_kwargs['drop_last'], False)
        test_eq(captured_kwargs['shuffle'], False)
        
        # Verify dataset wrapping and vocab patching
        wrapped_ds = captured_kwargs['dataset']
        test_eq(wrapped_ds.x_keys, "scan")
        test_eq(wrapped_ds.y_keys, "mask")
        test_eq(wrapped_ds.vocab, ["A", "B"])

        # --- 2. Test explicit overrides ---
        _create_test_dl(dummy_test_ds, mock_dls, batch_size=8, num_workers=1)
        test_eq(captured_kwargs['batch_size'], 8) # Successfully overridden
        test_eq(captured_kwargs['num_workers'], 1) # Successfully overridden

    finally:
        # Restore the original global environment perfectly
        for f in target_funcs:
            if orig_globals[f] is not None:
                globals()[f] = orig_globals[f]
            else:
                globals().pop(f, None)

    return "_create_test_dl routing tests passed"

test_eq(test_create_test_dl_routing(), "_create_test_dl routing tests passed")

In [ ]:
#| export
def test_biodataloader(dls:DataLoaders, 
                       test_data:str|Path|pd.DataFrame|MonaiDataset, 
                       with_labels=True, 
                       csv_header='infer', 
                       csv_delimiter=None, 
                       csv_quoting=0
                       ):
    """
    Create a test `DataLoader` from various data sources (DataFrame, CSV, Directory, or MONAI Dataset).
    
    Returns:
        A PyTorch/fastai DataLoader configured for testing/inference.
    """
    if isinstance(test_data, pd.DataFrame):
        # Handle DataFrame case directly
        test_dl = dls.test_dl(test_data, with_labels=with_labels)
    elif isinstance(test_data, (str, Path)):
        test_data = Path(test_data)
        # Check if it's a CSV file
        if test_data.suffix.lower() == '.csv':
            # Handle CSV file case
            df = pd.read_csv(test_data, header=csv_header, delimiter=csv_delimiter, quoting=csv_quoting)
            test_dl = dls.test_dl(df, with_labels=with_labels)
        else:
            # Handle non-CSV file case - get image files from directory
            test_dl = dls.test_dl(get_image_files(test_data), with_labels=with_labels)
    elif isinstance(test_data, MonaiDataset):
        test_dl = _create_test_dl(test_data, dls)
    
    return test_dl

In [ ]:
from unittest.mock import MagicMock
import pandas as pd

# Example: Generating a Test DataLoader dynamically
# test_biodataloader acts as a universal router. You can pass a DataFrame, 
# a CSV file path, an image directory, or a MONAI Dataset, and it will 
# automatically build the correct test DataLoader using your existing DataLoaders.

# 1. We use a mock DataLoaders object for the documentation example
mock_dls = MagicMock()
mock_dls.test_dl.return_value = "Ready-to-use Test DataLoader"

# 2. Define a simple test DataFrame
test_df = pd.DataFrame({"filename": ["test_01.nii", "test_02.nii"]})

try:
    # 3. Generate the test DataLoader directly from the DataFrame
    test_dl = test_biodataloader(
        dls=mock_dls,
        test_data=test_df,
        with_labels=False # Set to False if the test set lacks target columns
    )
    print(f"Result: {test_dl}")
except Exception as e:
    pass

Result: Ready-to-use Test DataLoader


In [ ]:
#| hide
from unittest.mock import patch, MagicMock
import pandas as pd
from pathlib import Path
from fastcore.test import test_eq

# 🛡️ ANTI-NAME-ERROR SHIELD:
# Safely mock MonaiDataset if it hasn't been imported into the global namespace yet.
if 'MonaiDataset' not in globals():
    class MonaiDataset: 
        def __init__(self, *args, **kwargs): pass # Accept arguments just like the real one

def test_test_biodataloader():
    mock_dls = MagicMock()
    mock_dls.test_dl.return_value = "Fastai_Test_DL"
    
    # --- 1. Test DataFrame Routing ---
    dummy_df = pd.DataFrame({"col": [1, 2]})
    res_df = test_biodataloader(mock_dls, dummy_df)
    test_eq(res_df, "Fastai_Test_DL")
    
    # --- 2. Test CSV Path Routing ---
    with patch('pandas.read_csv', return_value=dummy_df) as mock_read_csv:
        res_csv = test_biodataloader(mock_dls, "dummy_path.csv")
        test_eq(res_csv, "Fastai_Test_DL")
        test_eq(mock_read_csv.called, True)
        
    # --- 3. Test Directory Path Routing ---
    # Safely hijack get_image_files globally for this test
    orig_get_image_files = globals().get('get_image_files')
    globals()['get_image_files'] = lambda x: ["img1.png", "img2.png"]
    try:
        res_dir = test_biodataloader(mock_dls, "dummy_folder")
        test_eq(res_dir, "Fastai_Test_DL")
        test_eq(mock_dls.test_dl.call_args[0][0], ["img1.png", "img2.png"])
    finally:
        if orig_get_image_files: globals()['get_image_files'] = orig_get_image_files
        else: del globals()['get_image_files']
        
    # --- 4. Test MonaiDataset Routing ---
    orig_create_test_dl = globals().get('_create_test_dl')
    globals()['_create_test_dl'] = lambda ds, dls: "Monai_Test_DL"
    try:
        # We pass a dummy list [1] so the REAL MONAI Dataset doesn't crash 
        # due to the missing 'data' positional argument.
        dummy_monai = MonaiDataset([1])
        res_monai = test_biodataloader(mock_dls, dummy_monai)
        test_eq(res_monai, "Monai_Test_DL")
    finally:
        if orig_create_test_dl: globals()['_create_test_dl'] = orig_create_test_dl
        else: del globals()['_create_test_dl']
        
    return "test_biodataloader routing tests passed"

test_eq(test_test_biodataloader(), "test_biodataloader routing tests passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()